In [2]:
#qt for gui figures, inline for embedded in notebook 
%matplotlib qt
import hyperspy.api as hs
import exspy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [3]:
s = hs.load("JS006_0p31.rpl", stack=True)
s.set_signal_type("EDS_SEM")
s.metadata.General.title = 'Mulit-phase Jadeite Region'
s


  0%|          | 0/1 [00:00<?, ?it/s]

<EDSSEMSpectrum, title: Mulit-phase Jadeite Region, dimensions: (1024, 768|2665)>

In [4]:
s.axes_manager.gui() 
#Axis 0 and 1 tabs, enter microns for units, scale which is 0.31. For axis 3: enter units is keV, scale is 0.01, with a -0.96 offset

In [5]:
s = s.rebin((1024/8,768/8,2665))


In [6]:
#Crop is to remove Bruker strobe peak. 10keV was chosen because there are no further characteristic peaks >10keV for this systems
s_cropped = s.isig[0.2:10.0]

In [7]:
#PCA/ICA
s_cropped.change_dtype('float')
s_cropped.decomposition(normalize_poissonian_noise=True)

/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0. Use `RGB_DTYPES` instead
  warnings.warn(
/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0.
  warnings.warn(


Decomposition info:
  normalize_poissonian_noise=True
  algorithm=SVD
  output_dimension=None
  centre=None


In [8]:
s_cropped.plot_explained_variance_ratio()
#pop up:gui plot

<Axes: title={'center': 'Mulit-phase Jadeite Region\nPCA Scree Plot'}, xlabel='Principal component index', ylabel='Proportion of variance'>

In [9]:
s_cropped.change_dtype('float')
s_cropped.decomposition(algorithm='NMF', solver='mu', output_dimension=6, normalize_poissonian_noise=True, random_state=42)

Decomposition info:
  normalize_poissonian_noise=True
  algorithm=NMF
  output_dimension=6
  centre=None
scikit-learn estimator:
NMF(n_components=6, random_state=42, solver='mu')


In [10]:
s_cropped.plot_decomposition_results(True)
#pop up:gui plot

/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0.
  warnings.warn(


/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0.
  warnings.warn(
/Users/thomaslam/anaconda3/lib/python3.13/site-packages/rsciio/utils/rgb_tools.py:62: VisibleDeprecationWarning: The module `rsciio.utils.rgb_tools` has been renamed to `rsciio.utils.rgb` and it will be removed in version 1.0.
  warnings.warn(


In [11]:
from pathlib import Path

loadings = s_cropped.learning_results.loadings
height, width = 96, 128

outdir = Path("output")
outdir.mkdir(exist_ok=True)

for i in range(6):
    loading_map = loadings[:, i].reshape(height, width)

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(loading_map, cmap="gray")
    ax.axis("off")

    fig.savefig(
        outdir / f"nmf_component_{i}_gray.png",
        dpi=150,
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.close(fig)

    print(f"Saved nmf_component_{i}_gray.png")

Saved nmf_component_0_gray.png
Saved nmf_component_1_gray.png
Saved nmf_component_2_gray.png
Saved nmf_component_3_gray.png
Saved nmf_component_4_gray.png
Saved nmf_component_5_gray.png


In [12]:
from skimage.filters import threshold_otsu
from skimage import io
from skimage import io, transform

# Load component 0 
img_rgba = io.imread('output/nmf_component_0_gray.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply Otsu thresholding
threshold = threshold_otsu(img_96x128)
binary = img_96x128 >= threshold

# Get area fraction and save as variable C0_AFrac, where C0 is Component 0
C0_AFrac = binary.sum() / binary.size

# Save the binary mask as PNG
outdir = Path("output")
mask_uint8 = (binary * 255).astype(np.uint8)
mask_fname = outdir / "nmf_component_0_mask.png"
io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname}")

# Create figure with 3 panels (pop up gui)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap='gray')
axes[0].set_title(f'Component 0\nShape: {img_96x128.shape}')
axes[0].axis('off')

# Panel 2: Histogram with threshold line
axes[1].hist(img_96x128.flatten(), bins=256, color='gray', alpha=0.7)
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {threshold:.2f}')
axes[1].set_title(f'Histogram with Otsu Threshold\nArea Fraction: {C0_AFrac:.2%}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Panel 3: Binary mask (thresholded)
axes[2].imshow(binary, cmap='gray')
axes[2].set_title(f'Binary Mask\nForeground: {binary.sum()} pixels\nBackground: {(~binary).sum()} pixels')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"  Image shape: {img_96x128.shape}")
print(f"  Otsu threshold: {threshold:.2f}")
print(f"  C0_AFrac = {C0_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Converted RGBA to grayscale. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: output/nmf_component_0_mask.png
  Image shape: (96, 128)
  Otsu threshold: 172.19
  C0_AFrac = 88.94%
  Foreground pixels: 10929
  Background pixels: 1359
  Total pixels: 12288


In [13]:
# Load component 1
img_rgba = io.imread('output/nmf_component_1_gray.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply Otsu thresholding
threshold = threshold_otsu(img_96x128)
binary = img_96x128 >= threshold

# Get area fraction and save as variable C1_AFrac, where C1 is Component 1
C1_AFrac = binary.sum() / binary.size

# Save the binary mask as PNG
outdir = Path("output")
mask_uint8 = (binary * 255).astype(np.uint8)
mask_fname = outdir / "nmf_component_1_mask.png"
io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname}")

# Create figure with 3 panels (pop up gui)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap='gray')
axes[0].set_title(f'Component 1\nShape: {img_96x128.shape}')
axes[0].axis('off')

# Panel 2: Histogram with threshold line
axes[1].hist(img_96x128.flatten(), bins=256, color='gray', alpha=0.7)
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {threshold:.2f}')
axes[1].set_title(f'Histogram with Otsu Threshold\nArea Fraction: {C1_AFrac:.2%}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Panel 3: Binary mask (thresholded)
axes[2].imshow(binary, cmap='gray')
axes[2].set_title(f'Binary Mask\nForeground: {binary.sum()} pixels\nBackground: {(~binary).sum()} pixels')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"  Image shape: {img_96x128.shape}")
print(f"  Otsu threshold: {threshold:.2f}")
print(f"  C1_AFrac = {C1_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Converted RGBA to grayscale. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: output/nmf_component_1_mask.png
  Image shape: (96, 128)
  Otsu threshold: 80.92
  C1_AFrac = 9.19%
  Foreground pixels: 1129
  Background pixels: 11159
  Total pixels: 12288


In [14]:
# Load component 2
img_rgba = io.imread('output/nmf_component_2_gray.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply Otsu thresholding
threshold = threshold_otsu(img_96x128)
binary = img_96x128 >= threshold

# Get area fraction and save as variable C2_AFrac, where C2 is Component 2
C2_AFrac = binary.sum() / binary.size

# Save the binary mask as PNG
outdir = Path("output")
mask_uint8 = (binary * 255).astype(np.uint8)
mask_fname = outdir / "nmf_component_2_mask.png"
io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname}")

# Create figure with 3 panels (pop up gui)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap='gray')
axes[0].set_title(f'Component 2\nShape: {img_96x128.shape}')
axes[0].axis('off')

# Panel 2: Histogram with threshold line
axes[1].hist(img_96x128.flatten(), bins=256, color='gray', alpha=0.7)
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {threshold:.2f}')
axes[1].set_title(f'Histogram with Otsu Threshold\nArea Fraction: {C2_AFrac:.2%}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Panel 3: Binary mask (thresholded)
axes[2].imshow(binary, cmap='gray')
axes[2].set_title(f'Binary Mask\nForeground: {binary.sum()} pixels\nBackground: {(~binary).sum()} pixels')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"  Image shape: {img_96x128.shape}")
print(f"  Otsu threshold: {threshold:.2f}")
print(f"  C2_AFrac = {C2_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Converted RGBA to grayscale. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: output/nmf_component_2_mask.png
  Image shape: (96, 128)
  Otsu threshold: 60.89
  C2_AFrac = 1.59%
  Foreground pixels: 195
  Background pixels: 12093
  Total pixels: 12288


In [15]:
# Load component 3
img_rgba = io.imread('output/nmf_component_3_gray.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply Otsu thresholding
threshold = threshold_otsu(img_96x128)
binary = img_96x128 >= threshold

# Get area fraction and save as variable C3_AFrac, where C3 is Component 3
C3_AFrac = binary.sum() / binary.size

# Save the binary mask as PNG
outdir = Path("output")
mask_uint8 = (binary * 255).astype(np.uint8)
mask_fname = outdir / "nmf_component_3_mask.png"
io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname}")

# Create figure with 3 panels (pop up gui)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap='gray')
axes[0].set_title(f'Component 3\nShape: {img_96x128.shape}')
axes[0].axis('off')

# Panel 2: Histogram with threshold line
axes[1].hist(img_96x128.flatten(), bins=256, color='gray', alpha=0.7)
axes[1].axvline(threshold, color='red', linestyle='--', linewidth=2, 
                label=f'Threshold = {threshold:.2f}')
axes[1].set_title(f'Histogram with Otsu Threshold\nArea Fraction: {C3_AFrac:.2%}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Panel 3: Binary mask (thresholded)
axes[2].imshow(binary, cmap='gray')
axes[2].set_title(f'Binary Mask\nForeground: {binary.sum()} pixels\nBackground: {(~binary).sum()} pixels')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"  Image shape: {img_96x128.shape}")
print(f"  Otsu threshold: {threshold:.2f}")
print(f"  C3_AFrac = {C3_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Converted RGBA to grayscale. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: output/nmf_component_3_mask.png
  Image shape: (96, 128)
  Otsu threshold: 17.43
  C3_AFrac = 20.54%
  Foreground pixels: 2524
  Background pixels: 9764
  Total pixels: 12288


In [16]:
from skimage import io, transform
from skimage.filters import threshold_multiotsu

# Settings
N_CLASSES = 3
KEEP_CLASS = N_CLASSES - 1  # 0 = darkest, 1 = middle, 2 = brightest

# Load component 4
img_rgba = io.imread("output/nmf_component_4_gray.png")

# Grayscale PNG saved as RGB or RGBA: take the first channel
if img_rgba.ndim == 3:
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Extracted grayscale channel. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Resize to 96 x 128
img_96x128 = transform.resize(
    img,
    (96, 128),
    order=1,
    preserve_range=True,
    anti_aliasing=True,
).astype(np.float32)

print(f"Resized image shape: {img_96x128.shape}")

# Apply multi-Otsu directly to the original grayscale image
thresholds = threshold_multiotsu(
    img_96x128,
    classes=N_CLASSES,
)

# Assign pixels to classes: 0 = darkest ... N_CLASSES - 1 = brightest
regions = np.digitize(img_96x128, bins=thresholds)

# Keep the selected class
binary = regions == KEEP_CLASS

# Component 4 area fraction
C4_AFrac = binary.mean()

# Save binary mask as PNG
outdir = Path("output")
outdir.mkdir(parents=True, exist_ok=True)

mask_uint8 = binary.astype(np.uint8) * 255
mask_fname = outdir / "nmf_component_4_mask.png"

io.imsave(mask_fname, mask_uint8, check_contrast=False)
print(f"Saved mask: {mask_fname.resolve()}")

# Create figure with 4 panels
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

# Panel 1: Resized grayscale image
axes[0].imshow(img_96x128, cmap="gray")
axes[0].set_title(f"Component 4\nShape: {img_96x128.shape}")
axes[0].axis("off")

# Panel 2: Histogram with all threshold lines
axes[1].hist(
    img_96x128.ravel(),
    bins=256,
    color="gray",
    alpha=0.7,
)

for i, threshold in enumerate(thresholds):
    axes[1].axvline(
        threshold,
        color=f"C{i + 1}",
        linestyle="--",
        linewidth=2,
        label=f"T{i + 1} = {threshold:.2f}",
    )

axes[1].set_title(
    f"Multi-Otsu ({N_CLASSES} classes)\n"
    f"Class {KEEP_CLASS} Area Fraction: {C4_AFrac:.2%}"
)
axes[1].set_xlabel("Pixel Intensity")
axes[1].set_ylabel("Frequency")
axes[1].legend()

# Panel 3: Multi-Otsu class map
class_plot = axes[2].imshow(
    regions,
    cmap=plt.get_cmap("viridis", N_CLASSES),
    vmin=-0.5,
    vmax=N_CLASSES - 0.5,
    interpolation="nearest",
)
axes[2].set_title("Class Map\n0 = darkest; highest class = brightest")
axes[2].axis("off")

fig.colorbar(
    class_plot,
    ax=axes[2],
    ticks=np.arange(N_CLASSES),
    fraction=0.046,
    pad=0.04,
)

# Panel 4: Binary mask
axes[3].imshow(
    binary,
    cmap="gray",
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
axes[3].set_title(
    f"Binary Mask — Class {KEEP_CLASS}\n"
    f"Foreground: {binary.sum()} pixels\n"
    f"Background: {(~binary).sum()} pixels"
)
axes[3].axis("off")

plt.tight_layout()
plt.show()

# Report
print(f"  Image shape: {img_96x128.shape}")
print(f"  Multi-Otsu thresholds: {np.round(thresholds, 2)}")

for k in range(N_CLASSES):
    class_count = np.count_nonzero(regions == k)
    print(
        f"  Class {k}: {class_count} pixels "
        f"({class_count / regions.size:.2%})"
    )

print(f"  Selected class: {KEEP_CLASS}")
print(f"  C4_AFrac = {C4_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

Extracted grayscale channel. Shape: (871, 1162)
Resized image shape: (96, 128)
Saved mask: /Users/thomaslam/Downloads/output/nmf_component_4_mask.png
  Image shape: (96, 128)
  Multi-Otsu thresholds: [17.54 66.8 ]
  Class 0: 7322 pixels (59.59%)
  Class 1: 4869 pixels (39.62%)
  Class 2: 97 pixels (0.79%)
  Selected class: 2
  C4_AFrac = 0.79%
  Foreground pixels: 97
  Background pixels: 12191
  Total pixels: 12288


In [17]:
# Load masks
c1 = io.imread("output/nmf_component_1_mask.png")
c3 = io.imread("output/nmf_component_3_mask.png")

# Defining masks for white foreground / black background:
c1_fg = c1 > 128
c3_fg = c3 > 128

# Shared foreground
shared = c1_fg & c3_fg

# Make shared area black in Component 3
c3_new = c3.copy()
c3_new[shared] = 0

# Save result
outpath = Path("output/nmf_component_3_mask_blackshared.png")
io.imsave(outpath, c3_new.astype(np.uint8), check_contrast=False)

print(f"Saved: {outpath}")
print(f"Shared pixels made black: {shared.sum()}")

Saved: output/nmf_component_3_mask_blackshared.png
Shared pixels made black: 829


In [18]:
from skimage import io, morphology

# Denoise Component 3
c3 = io.imread("output/nmf_component_3_mask_blackshared.png")

# white = foreground
c3_fg = c3 > 128

# stronger cleanup
c3_clean = morphology.remove_small_objects(c3_fg, min_size=50, connectivity=2)
c3_clean = morphology.remove_small_holes(c3_clean, area_threshold=50)

# save back with white foreground
c3_out = np.where(c3_clean, 255, 0).astype(np.uint8)
io.imsave(Path("output/nmf_component_3_mask_clean.png"), c3_out, check_contrast=False)

print(f"Foreground pixels before: {c3_fg.sum()}")
print(f"Foreground pixels after:  {c3_clean.sum()}")

Foreground pixels before: 1695
Foreground pixels after:  681


In [19]:
# Corrected component 3 area fraction calc
img_rgba = io.imread('output/nmf_component_3_mask_clean.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Apply Otsu thresholding
threshold = threshold_otsu(img)
binary = img >= threshold
# Get area fraction and save as variable C_C3_AFrac, where C_C3 is the Corrected Component 3
C_C3_AFrac = binary.sum() / binary.size

print(f"  C_C3_AFrac = {C_C3_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

  C_C3_AFrac = 5.54%
  Foreground pixels: 681
  Background pixels: 11607
  Total pixels: 12288


In [20]:
# Applying C1 mask to original data cube prior to NMF 

cube = s.rebin((1024/8, 768/8, 2665)).data

mask = io.imread("output/nmf_component_1_mask.png")
keep = mask > 128

cube1 = cube.reshape(-1, cube.shape[-1])
keep1 = keep.reshape(-1)

sum_spec_array = cube1[keep1].sum(axis=0)

# energy axis from original signal
x = s.axes_manager.signal_axes[0].axis

plt.figure(figsize=(8, 4))
plt.plot(x, sum_spec_array, lw=1)
plt.xlim(0.2, 7)
plt.xlabel(f"{s.axes_manager.signal_axes[0].name} ({s.axes_manager.signal_axes[0].units})")
plt.ylabel("Counts")
plt.title("Component 1 summed spectrum")

plt.tight_layout()
plt.show()

In [21]:
from lmfit.models import GaussianModel, LinearModel, QuadraticModel

# Deconvolution and background removal for Component 1
# Energy axis from the HyperSpy signal object
energy_keV = np.asarray(
    s.axes_manager.signal_axes[0].axis,
    dtype=float
)

# Counts from the NumPy summed-spectrum array
counts = np.asarray(
    sum_spec_array,
    dtype=float
).squeeze()

# If the energy axis is in eV rather than keV, convert it.
# Comment this out if energy_keV is already in keV.
if np.nanmax(energy_keV) > 100:
    energy_keV = energy_keV / 1000.0

if energy_keV.ndim != 1:
    raise ValueError(
        f"Energy axis must be 1D; got shape {energy_keV.shape}"
    )

if counts.ndim != 1:
    raise ValueError(
        f"Counts must be 1D; got shape {counts.shape}"
    )

if energy_keV.size != counts.size:
    raise ValueError(
        f"Energy axis has {energy_keV.size} channels, "
        f"but counts has {counts.size} values."
    )

print("Type of sum_spec_array:", type(sum_spec_array))
print("Energy-axis shape:", energy_keV.shape)
print("Counts shape:", counts.shape)
print(
    "Energy range:",
    energy_keV.min(),
    "to",
    energy_keV.max(),
    "keV"
)

# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "O_Ka": 0.525
}


# ============================================================
# Fit O peaks plus local continuum
# ============================================================

def fit_low_energy_eds(
    energy_keV,
    counts,
    e_min=0.3,
    e_max=0.7,
    fwhm_eV=100.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Fit O Kα,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_low_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "O_Ka": "tab:blue"
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "O EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_low_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=0.3,
    e_max=0.7,
    fwhm_eV=100.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
O_net = areas.loc[
    areas["line"].eq("O_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("O_net:", O_net)



# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit ((pop up gui)
fig = plot_low_energy_fit(
    result
)

plt.show()

Type of sum_spec_array: <class 'numpy.ndarray'>
Energy-axis shape: (2665,)
Counts shape: (2665,)
Energy range: -0.96 to 25.68 keV
[[Model]]
    (Model(linear, prefix='bg_') + Model(gaussian, prefix='O_Ka_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 65
    # data points      = 40
    # variables        = 4
    chi-square         = 4754.00121
    reduced chi-square = 132.055589
    Akaike info crit   = 199.114498
    Bayesian info crit = 205.870016
    R-squared          = 0.98460134
[[Variables]]
    bg_slope:       -12.8977768 +/- 313.143631 (2427.89%) (init = -617.9487)
    bg_intercept:    313.641131 +/- 152.115454 (48.50%) (init = 902.3846)
    O_Ka_amplitude:  1210.90730 +/- 41.4977293 (3.43%) (init = 1178.645)
    O_Ka_center:     0.525 (fixed)
    O_Ka_sigma:      0.02547965 +/- 7.1484e-04 (2.81%) (init = 0.04246609)
    O_Ka_fwhm:       0.06000000 +/- 0.00168333 (2.81%) == '2.3548200*O_Ka_sigma'
    O_Ka_height:     18959.5248 +/- 795.63

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,O_Ka,1210.907296,121090.729559,41.497729,0.525,25.479654,60.0


O_net: 1210.9072955924973
Continuum-subtracted spectrum generated: (40,)


In [22]:
# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "Na_Ka": 1.041,
    "Mg_Ka": 1.254,
    "Al_Ka": 1.487,
    "Si_Ka": 1.740,
}


# ============================================================
# Fit Na, Mg, Al, Si peaks plus local continuum
# ============================================================

def fit_low_energy_eds(
    energy_keV,
    counts,
    e_min=0.90,
    e_max=1.90,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Simultaneously fit Na Kα, Mg Kα, Al Kα, Si Kα,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_low_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "Na_Ka": "tab:blue",
        "Mg_Ka": "tab:red",
        "Al_Ka": "tab:green",
        "Si_Ka": "tab:orange",
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "Na–Mg–Al–Si simultaneous EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_low_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=0.90,
    e_max=1.90,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
Na_net = areas.loc[
    areas["line"].eq("Na_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Mg_net = areas.loc[
    areas["line"].eq("Mg_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Al_net = areas.loc[
    areas["line"].eq("Al_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Si_net = areas.loc[
    areas["line"].eq("Si_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("Si_net:", Si_net)
print("Mg_net:", Mg_net)
print("Al_net:", Al_net)
print("Si_net:", Si_net)


# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit ((pop up gui)
fig = plot_low_energy_fit(
    result
)

plt.show()

[[Model]]
    ((((Model(linear, prefix='bg_') + Model(gaussian, prefix='Na_Ka_')) + Model(gaussian, prefix='Mg_Ka_')) + Model(gaussian, prefix='Al_Ka_')) + Model(gaussian, prefix='Si_Ka_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 143
    # data points      = 101
    # variables        = 10
    chi-square         = 2675.69098
    reduced chi-square = 29.4031976
    Akaike info crit   = 350.961084
    Bayesian info crit = 377.112290
    R-squared          = 0.99832370
[[Variables]]
    bg_slope:         515.456041 +/- 85.9125290 (16.67%) (init = 852)
    bg_intercept:    -56.2925387 +/- 122.717328 (218.00%) (init = 0)
    Na_Ka_amplitude:  566.185060 +/- 16.1668279 (2.86%) (init = 494.6952)
    Na_Ka_center:     1.041 (fixed)
    Na_Ka_sigma:      0.03439753 +/- 8.8157e-04 (2.56%) (init = 0.05732922)
    Mg_Ka_amplitude:  373.912088 +/- 13.5672126 (3.63%) (init = 301.3448)
    Mg_Ka_center:     1.254 (fixed)
    Mg_Ka_sigma:      0.03439753 +/- 

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,Na_Ka,566.185060,56618.505991,16.166828,1.041,34.397534,81.000000
1,Mg_Ka,373.912088,37391.208789,13.567213,1.254,34.397534,81.000000
2,Al_Ka,820.973883,82097.388258,18.045014,1.487,34.397534,81.000000
3,Si_Ka,2703.840818,270384.081781,30.579256,1.740,34.614500,81.510916


Si_net: 2703.840817809544
Mg_net: 373.9120878923477
Al_net: 820.9738825774498
Si_net: 2703.840817809544
Continuum-subtracted spectrum generated: (101,)


In [23]:

# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "Al_Si_sum": 3.23,
    "K_Ka": 3.312,
    "Si_Si_sum": 3.48,
    "K_Kb": 3.589,
    "Ca_Ka": 3.690,
    "Ca_Kb": 4.012
}


# ============================================================
# Fit Ca Ka and Kb peaks plus local continuum
# ============================================================

def fit_energy_eds(
    energy_keV,
    counts,
    e_min=3.0,
    e_max=4.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Simultaneously fit Ca Kα, Ca Kb,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "Al_Si_sum": "tab:purple",
        "K_Ka": "tab:blue",
        "Si_Si_sum": "tab:orange",
        "K_Kb": "tab:blue",
        "Ca_Ka": "tab:green",
        "Ca_Kb": "tab:green",
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "simultaneous EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=3.0,
    e_max=4.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
Ca_net = areas.loc[
    areas["line"].eq("Ca_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("Ca_net:", Ca_net)


# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit (pop up gui)
fig = plot_energy_fit(
    result
)

plt.show()

[[Model]]
    ((((((Model(linear, prefix='bg_') + Model(gaussian, prefix='Al_Si_sum_')) + Model(gaussian, prefix='K_Ka_')) + Model(gaussian, prefix='Si_Si_sum_')) + Model(gaussian, prefix='K_Kb_')) + Model(gaussian, prefix='Ca_Ka_')) + Model(gaussian, prefix='Ca_Kb_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 408
    # data points      = 151
    # variables        = 14
    chi-square         = 242.246356
    reduced chi-square = 1.76822158
    Akaike info crit   = 99.3739812
    Bayesian info crit = 141.615899
    R-squared          = 0.99744475
[[Variables]]
    bg_slope:            -88.1229598 +/- 7.86450117 (8.92%) (init = -87.33333)
    bg_intercept:         597.216000 +/- 32.9889841 (5.52%) (init = 598.5)
    Al_Si_sum_amplitude:  3.34492351 +/- 2.35217111 (70.32%) (init = 2.384533)
    Al_Si_sum_center:     3.23 (fixed)
    Al_Si_sum_sigma:      0.03957155 +/- 0.02104299 (53.18%) (init = 0.05732922)
    K_Ka_amplitude:       2.2901e-06 +/

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,Al_Si_sum,3.344924e+00,334.492351,2.352171,3.230,39.571553,93.183885
1,K_Ka,2.290061e-06,0.000229,7.146914,3.312,103.187208,242.987301
2,Si_Si_sum,4.127873e+00,412.787280,4.176418,3.480,43.935060,103.459159
3,K_Kb,1.319752e-07,0.000013,6.007024,3.589,103.192595,242.999987
4,Ca_Ka,4.270787e+02,42707.871058,4.131182,3.690,44.809712,105.518806
5,Ca_Kb,5.194501e+01,5194.501197,1.654241,4.012,47.540922,111.950314


Ca_net: 427.0787105830813
Continuum-subtracted spectrum generated: (151,)


In [24]:
# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "Mn_Ka": 5.893,
    "Mn_Kb": 6.491,
    "Fe_Ka": 6.397,
    "Fe_Kb": 7.058
}


# ============================================================
# Fit Mn Ka, Mn Kb, Fe Ka, Fe Kb peaks plus local continuum
# ============================================================

def fit_energy_eds(
    energy_keV,
    counts,
    e_min=5.5,
    e_max=7.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Simultaneously fit Mn Ka, Mn Kb, Fe Kα, Fe Kb,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "Mn_Ka": "tab:blue",
        "Mn_Kb": "tab:blue",
        "Fe_Ka": "tab:brown",
        "Fe_Kb": "tab:brown"
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "simultaneous EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=5.5,
    e_max=7.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
Mn_net = areas.loc[
    areas["line"].eq("Mn_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Fe_net = areas.loc[
    areas["line"].eq("Fe_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("Mn_net:", Mn_net)
print("Fe_net:", Fe_net)


# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit (pop up gui)
fig = plot_energy_fit(
    result
)

plt.show()

[[Model]]
    ((((Model(linear, prefix='bg_') + Model(gaussian, prefix='Mn_Ka_')) + Model(gaussian, prefix='Mn_Kb_')) + Model(gaussian, prefix='Fe_Ka_')) + Model(gaussian, prefix='Fe_Kb_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 99
    # data points      = 200
    # variables        = 10
    chi-square         = 209.808080
    reduced chi-square = 1.10425305
    Akaike info crit   = 29.5751680
    Bayesian info crit = 62.5583416
    R-squared          = 0.97586473
[[Variables]]
    bg_slope:        -32.9152014 +/- 1.57831800 (4.80%) (init = -32.91457)
    bg_intercept:     322.727788 +/- 10.4968969 (3.25%) (init = 330.5302)
    Mn_Ka_amplitude:  2.35590344 +/- 0.67742355 (28.75%) (init = 1.660302)
    Mn_Ka_center:     5.893 (fixed)
    Mn_Ka_sigma:      0.04128941 +/- 0.01239514 (30.02%) (init = 0.05732922)
    Mn_Kb_amplitude:  0.23020470 +/- 0.83685872 (363.53%) (init = 12.6594)
    Mn_Kb_center:     6.491 (fixed)
    Mn_Kb_sigma:      0.0

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,Mn_Ka,2.355903,235.590344,0.677424,5.893,41.289409,97.229125
1,Mn_Kb,0.230205,23.020470,0.836859,6.491,34.397534,81.000000
2,Fe_Ka,51.553248,5155.324816,1.321025,6.397,58.291562,137.266135
3,Fe_Kb,5.755454,575.545400,0.706213,7.058,55.100003,129.750589


Mn_net: 2.3559034406946697
Fe_net: 51.5532481581767
Continuum-subtracted spectrum generated: (200,)


In [25]:
#Mn yielded 3.49 > the area standard, however visual detection is very bordeline and quantification was not pursued

# k-ratios corrected for the area fraction of the phase being quantified
std_df = pd.read_csv("15kV_Stds_Lib.csv")
std_row = std_df.iloc[0]

# Enter area fraction of the phase of interest (e.g., from the Otsu/multi-Otsu mask)
PHASE_AFRAC = 0.0919          # or: PHASE_AFRAC = C1_AFrac

if not np.isfinite(PHASE_AFRAC) or not (0 < PHASE_AFRAC <= 1):
    raise ValueError(f"Invalid phase area fraction ({PHASE_AFRAC}); must be in (0, 1].")

# Standard net counts (zero-based column positions in 15kV_Stds_Lib.csv)
std_net_counts = {
    "O":  pd.to_numeric(std_row.iloc[12], errors="raise"),
    "Na": pd.to_numeric(std_row.iloc[1],  errors="raise"),
    "Mg": pd.to_numeric(std_row.iloc[2],  errors="raise"),
    "Al": pd.to_numeric(std_row.iloc[3],  errors="raise"),
    "Si": pd.to_numeric(std_row.iloc[4],  errors="raise"),
    "Ca": pd.to_numeric(std_row.iloc[8],  errors="raise"),
    "Fe": pd.to_numeric(std_row.iloc[11], errors="raise")
}

unknown_net_counts = {
    "O":  O_net,
    "Na": Na_net,
    "Mg": Mg_net,
    "Al": Al_net,
    "Si": Si_net,
    "Ca": Ca_net,
    "Fe": Fe_net
}

# Every element must have both a standard and an unknown value
if set(std_net_counts) != set(unknown_net_counts):
    raise ValueError(
        f"Element mismatch: standards {sorted(std_net_counts)} vs "
        f"unknowns {sorted(unknown_net_counts)}"
    )

# Validate inputs
for element, i_std in std_net_counts.items():
    if not np.isfinite(i_std) or i_std <= 0:
        raise ValueError(f"{element}: invalid standard net count ({i_std}).")
    i_unknown = float(unknown_net_counts[element])
    if not np.isfinite(i_unknown) or i_unknown < 0:
        raise ValueError(f"{element}: invalid unknown net count ({i_unknown}).")

# k = I_unknown / (I_standard * area fraction)
kr_df = pd.DataFrame(
    {
        "element": list(std_net_counts.keys()),
        "k_ratio": [
            float(unknown_net_counts[el]) / (float(std_net_counts[el]) * PHASE_AFRAC)
            for el in std_net_counts
        ],
    }
)

print(f"Measured k-ratios (phase area fraction = {PHASE_AFRAC:.4f})")
display(kr_df.style.format({"k_ratio": "{:.6f}"}))

# Export for NeXL
input_csv = Path("input_kratios.csv").resolve()
kr_df[["element", "k_ratio"]].to_csv(input_csv, index=False, float_format="%.6f")
print(f"Saved: {input_csv}")

Measured k-ratios (phase area fraction = 0.0919)


,element,k_ratio
0,O,1.148091
1,Na,0.796040
2,Mg,0.052690
3,Al,0.803293
4,Si,0.843181
5,Ca,0.203583
6,Fe,0.031962


Saved: /Users/thomaslam/Downloads/input_kratios.csv


In [26]:
from pathlib import Path

workdir = Path("/Users/thomaslam/Documents/NeXL_XPP")
workdir.mkdir(exist_ok=True)

input_csv = workdir / "input_kratios.csv"
kr_df.to_csv(input_csv, index=False)

print(f"Saved NeXL input: {input_csv}")
print(kr_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

Saved NeXL input: /Users/thomaslam/Documents/NeXL_XPP/input_kratios.csv
element  k_ratio
      O 1.148091
     Na 0.796040
     Mg 0.052690
     Al 0.803293
     Si 0.843181
     Ca 0.203583
     Fe 0.031962


In [27]:
from pathlib import Path
import subprocess
import textwrap
import pandas as pd

# ================================================================
# 1. Python-controlled inputs
# ================================================================
BEAM_ENERGY_KEV = 15.0
TAKE_OFF_ANGLE_DEG = 35.0

input_csv = Path(input_csv).resolve()
standards_csv = Path("/Users/thomaslam/Documents/NeXL_XPP/Standards_df.csv")

project_dir = Path("/Users/thomaslam/Documents/NeXL_XPP")
project_dir.mkdir(parents=True, exist_ok=True)

results_csv = project_dir / "xpp_results.csv"
julia_script = project_dir / "run_xpp_from_python.jl"

# ================================================================
# 2. Validate Python inputs and write k-ratio CSV
# ================================================================
if not input_csv.is_file():
    raise FileNotFoundError(f"Python cannot find:\n{input_csv}")

if not standards_csv.is_file():
    raise FileNotFoundError(f"Python cannot find:\n{standards_csv}")

kr_df = kr_df.copy()
kr_df.columns = kr_df.columns.astype(str).str.strip()

required_columns = {"element", "k_ratio"}
if not required_columns.issubset(set(kr_df.columns)):
    raise ValueError(
        f"kr_df must contain {required_columns}; found {list(kr_df.columns)}"
    )

kr_df["element"] = kr_df["element"].astype(str).str.strip()
kr_df["k_ratio"] = pd.to_numeric(kr_df["k_ratio"], errors="raise")

if kr_df["element"].duplicated().any():
    duplicates = kr_df.loc[
        kr_df["element"].duplicated(keep=False), "element"
    ].tolist()
    raise ValueError(
        "Each element must appear only once in kr_df. "
        f"Duplicate entries: {duplicates}"
    )

if (kr_df["k_ratio"] < 0).any():
    raise ValueError("All k_ratio values must be non-negative.")

print("Python input k-ratios:")
display(kr_df[["element", "k_ratio"]])

kr_df[["element", "k_ratio"]].to_csv(input_csv, index=False)

# ================================================================
# 3. Julia backend written by Python
# ================================================================
julia_code = r'''
using CSV
using DataFrames
using NeXLMatrixCorrection

input_kratios_csv = ARGS[1]
standards_csv = ARGS[2]
results_csv = ARGS[3]
beam_energy_kev = parse(Float64, ARGS[4])
take_off_angle_deg = parse(Float64, ARGS[5])

kr_df = CSV.read(input_kratios_csv, DataFrame)
standards_df = CSV.read(standards_csv, DataFrame)

required_columns = ["element", "k_ratio"]
found_columns = String.(names(kr_df))
missing_columns = setdiff(required_columns, found_columns)

isempty(missing_columns) || error(
    "input_kratios.csv missing columns: $(missing_columns). " *
    "Found: $(found_columns)"
)

println(
    "Quantification conditions from Python: E0 = ", beam_energy_kev,
    " keV; TOA = ", take_off_angle_deg, " degrees"
)

unknown_properties = Dict(
    :BeamEnergy => beam_energy_kev * 1.0e3,
    :TakeOffAngle => deg2rad(take_off_angle_deg)
)

standard_properties = Dict(
    :BeamEnergy => beam_energy_kev * 1.0e3,
    :TakeOffAngle => deg2rad(take_off_angle_deg)
)

const ATOMIC_WEIGHT = Dict(
    "O"  => 15.999,
    "Na" => 22.990,
    "Mg" => 24.305,
    "Al" => 26.982,
    "Si" => 28.086,
    "K"  => 39.098,
    "Ca" => 40.078,
    "Ti" => 47.867,
    "Fe" => 55.845
)

const OXIDE_DEFINITIONS = Dict(
    :SiO2  => ("Si", 1, 2),
    :TiO2  => ("Ti", 1, 2),
    :Al2O3 => ("Al", 2, 3),
    :FeO   => ("Fe", 1, 1),
    :Fe2O3 => ("Fe", 2, 3),
    :MgO   => ("Mg", 1, 1),
    :CaO   => ("Ca", 1, 1),
    :Na2O  => ("Na", 2, 1),
    :K2O   => ("K", 2, 1)
)

function elemental_fractions_from_oxides(row)
    elemental_wtpercent = Dict{String, Float64}()

    for (oxide, (element, n_element, n_oxygen)) in OXIDE_DEFINITIONS
        oxide in propertynames(row) || continue

        wtpercent = row[oxide]

        if ismissing(wtpercent) || Float64(wtpercent) <= 0.0
            continue
        end

        wtpercent = Float64(wtpercent)

        molecular_weight =
            n_element * ATOMIC_WEIGHT[element] +
            n_oxygen * ATOMIC_WEIGHT["O"]

        elemental_wtpercent[element] =
            get(elemental_wtpercent, element, 0.0) +
            wtpercent * n_element * ATOMIC_WEIGHT[element] / molecular_weight

        elemental_wtpercent["O"] =
            get(elemental_wtpercent, "O", 0.0) +
            wtpercent * n_oxygen * ATOMIC_WEIGHT["O"] / molecular_weight
    end

    total = sum(values(elemental_wtpercent))

    total > 0.0 || error(
        "No positive oxide wt% values found for standard: " *
        String(row.standard_id)
    )

    return Dict(
        element => wtpercent / total
        for (element, wtpercent) in elemental_wtpercent
    )
end

function get_standard_fractions(standard_id)
    hits = filter(
        row -> strip(String(row.standard_id)) == standard_id,
        standards_df
    )

    nrow(hits) == 1 || error(
        "Expected exactly one standard row for \"" * standard_id *
        "\"; found $(nrow(hits))."
    )

    return elemental_fractions_from_oxides(hits[1, :])
end

function make_material(standard_id)
    fractions = get_standard_fractions(standard_id)

    composition = join(
        [
            string(value, "*", element)
            for (element, value) in sort(collect(fractions); by=first)
        ],
        "+"
    )

    return parse(Material, composition; name=standard_id)
end

standards = Dict(
    "O"  => make_material("orthoclase_spi_mci"),
    "Na" => make_material("albite_spi_mci"),
    "Mg" => make_material("MgO_spi_mci"),
    "Al" => make_material("orthoclase_spi_mci"),
    "Si" => make_material("orthoclase_spi_mci"),
    "K" => make_material("orthoclase_spi_mci"),
    "Ca" => make_material("wollastonite_spi_mci"),
    "Mn" => make_material("rhodonite_spi_mci"),
    "Fe" => make_material("hematite_spi_mci")
)

lines = Dict(
    "O"  => n"O K-L3",
    "Na" => n"Na K-L3",
    "Mg" => n"Mg K-L3",
    "Al" => n"Al K-L3",
    "Si" => n"Si K-L3",
    "K" => n"K K-L3",
    "Ca" => n"Ca K-L3",
    "Mn" => n"Mn K-L3",
    "Fe" => n"Fe K-L3"
)

krs = KRatio[]

for row in eachrow(kr_df)
    element = strip(String(row.element))

    haskey(standards, element) || error(
        "No standard mapping for element: \"$element\""
    )

    k = Float64(row.k_ratio)
    k >= 0.0 || error("Negative k-ratio for $element")

    # Uses a 1% relative k-ratio uncertainty, with a small positive floor.
    sigma_k = max(0.01 * max(k, 1.0e-6), 1.0e-6)

    push!(
        krs,
        KRatio(
            [lines[element]],
            unknown_properties,
            standard_properties,
            standards[element],
            uv(k, sigma_k)
        )
    )
end

println("K-ratios received from Python:")
foreach(println, krs)

# NeXL quantification has completed successfully in this environment.
result = quantify(label("python_xpp"), krs)

# ----------------------------------------------------------------
# Export directly from IterationResult; do NOT call asa(DataFrame,...)
# because the installed NeXL DataFrames extension references _ZAF,
# which is undefined in that package version.
# ----------------------------------------------------------------

# `result.comp` is the final quantified Material. Build the CSV in
# input-element order, so Python receives a predictable tidy table.
result_elements = String[]
mass_fractions = Float64[]

for row in eachrow(kr_df)
    element_name = strip(String(row.element))

    # `parse(Element, ...)` creates the element key used by Material.
    element = parse(Element, element_name)

    # The Material is indexed by Element and returns the elemental
    # mass fraction. This bypasses NeXL's broken DataFrames extension.
    mass_fraction = Float64(result.comp[element])

    push!(result_elements, element_name)
    push!(mass_fractions, mass_fraction)
end

out_df = DataFrame(
    label = fill(string(result.label), length(result_elements)),
    converged = fill(Bool(result.converged), length(result_elements)),
    iterations = fill(Int(result.iterations), length(result_elements)),
    beam_energy_kev = fill(beam_energy_kev, length(result_elements)),
    take_off_angle_deg = fill(take_off_angle_deg, length(result_elements)),
    element = result_elements,
    mass_fraction = mass_fractions,
    wt_percent = 100.0 .* mass_fractions
)

CSV.write(results_csv, out_df)

println("Final quantification result: ", result)
println("Wrote results to: ", results_csv)
'''

julia_script.write_text(
    textwrap.dedent(julia_code),
    encoding="utf-8"
)

# ================================================================
# 4. Python launches Julia and reads results back
# ================================================================
run = subprocess.run(
    [
        "julia",
        str(julia_script),
        str(input_csv),
        str(standards_csv),
        str(results_csv),
        str(BEAM_ENERGY_KEV),
        str(TAKE_OFF_ANGLE_DEG),
    ],
    text=True,
    capture_output=True,
)

print("----- Julia stdout -----")
print(run.stdout)

if run.returncode != 0:
    print("----- Julia stderr -----")
    print(run.stderr)
    raise RuntimeError("Julia/NeXL quantification did not complete.")

if not results_csv.is_file():
    raise FileNotFoundError(
        f"Julia did not create the results output:\n{results_csv}"
    )

# ----- Load the Julia/NeXL results CSV back into Python -----
xpp_results = pd.read_csv(results_csv)

print("----- Results returned to Python -----")
display(xpp_results)

print("\n----- Quantified elemental total -----")
print(f"{xpp_results['mass_fraction'].sum():.6f}")

Python input k-ratios:


,element,k_ratio
0,O,1.148091
1,Na,0.796040
2,Mg,0.052690
3,Al,0.803293
4,Si,0.843181
5,Ca,0.203583
6,Fe,0.031962


----- Julia stdout -----
Quantification conditions from Python: E0 = 15.0 keV; TOA = 35.0 degrees
K-ratios received from Python:
k[O K-L3, orthoclase_spi_mci] = 1.148 ± 0.011
k[Na K-L3, albite_spi_mci] = 0.7960 ± 0.0080
k[Mg K-L3, MgO_spi_mci] = 0.0527 ± 0.0005
k[Al K-L3, orthoclase_spi_mci] = 0.8033 ± 0.0080
k[Si K-L3, orthoclase_spi_mci] = 0.8432 ± 0.0084
k[Ca K-L3, wollastonite_spi_mci] = 0.2036 ± 0.0020
k[Fe K-L3, hematite_spi_mci] = 0.0320 ± 0.0003
Final quantification result: Converged to python_xpp[O=0.4595,Na=0.0722,Mg=0.0391,Al=0.0785,Si=0.2698,Ca=0.0724,Fe=0.0254] in 6 steps.
Wrote results to: /Users/thomaslam/Documents/NeXL_XPP/xpp_results.csv

----- Results returned to Python -----


,label,converged,iterations,beam_energy_kev,take_off_angle_deg,element,mass_fraction,wt_percent
0,python_xpp,True,6,15.0,35.0,O,0.459482,45.948177
1,python_xpp,True,6,15.0,35.0,Na,0.072156,7.215605
2,python_xpp,True,6,15.0,35.0,Mg,0.039126,3.912610
3,python_xpp,True,6,15.0,35.0,Al,0.078527,7.852688
4,python_xpp,True,6,15.0,35.0,Si,0.269831,26.983089
5,python_xpp,True,6,15.0,35.0,Ca,0.072397,7.239739
6,python_xpp,True,6,15.0,35.0,Fe,0.025392,2.539235



----- Quantified elemental total -----
1.016911


In [28]:
# ================================================================
# 5. Convert quantified elemental mass fractions to oxide equivalents
# ================================================================

# Select the reporting convention for iron:
# "FeO"   = report all quantified Fe as ferrous iron
# "Fe2O3" = report all quantified Fe as ferric iron
IRON_OXIDE = "FeO"

# Atomic weights in g/mol
ATOMIC_WEIGHT = {
    "O": 15.999,
    "Na": 22.990,
    "Mg": 24.305,
    "Al": 26.982,
    "Si": 28.086,
    "Ca": 40.078,
    "Mn": 54.938,
    "Fe": 55.845,
}

# Oxide stoichiometry:
# oxide_name: (cation_symbol, number_of_cations, number_of_oxygens)
OXIDE_DEFINITIONS = {
    "Na2O": ("Na", 2, 1),
    "Al2O3": ("Al", 2, 3),
    "MgO": ("Mg", 1, 1),
    "SiO2": ("Si", 1, 2),
    "CaO": ("Ca", 1, 1),
    "MnO": ("Mn", 1, 1),
    IRON_OXIDE: ("Fe", 1, 1) if IRON_OXIDE == "FeO" else ("Fe", 2, 3),
}

# Element wt% -> oxide wt% factor:
# factor = molar_mass(oxide) / mass_of_cations_in_one_formula_unit
oxide_rows = []

for oxide, (element, n_cation, n_oxygen) in OXIDE_DEFINITIONS.items():
    match = xpp_results.loc[
        xpp_results["element"].astype(str).str.strip() == element
    ]

    if match.empty:
        print(f"Warning: no quantified {element} row; skipping {oxide}.")
        continue

    elemental_mass_fraction = float(match["mass_fraction"].iloc[0])
    elemental_wt_percent = 100.0 * elemental_mass_fraction

    oxide_molar_mass = (
        n_cation * ATOMIC_WEIGHT[element]
        + n_oxygen * ATOMIC_WEIGHT["O"]
    )

    cation_mass_per_formula_unit = n_cation * ATOMIC_WEIGHT[element]

    element_to_oxide_factor = (
        oxide_molar_mass / cation_mass_per_formula_unit
    )

    oxide_wt_percent = elemental_wt_percent * element_to_oxide_factor

    oxide_rows.append(
        {
            "oxide": oxide,
            "source_element": element,
            "element_mass_fraction": elemental_mass_fraction,
            "element_wt_percent": elemental_wt_percent,
            "element_to_oxide_factor": element_to_oxide_factor,
            "oxide_wt_percent": oxide_wt_percent,
            "oxide_mass_fraction": oxide_wt_percent / 100.0,
        }
    )

oxide_results = pd.DataFrame(oxide_rows)

print("----- Oxide-equivalent results (unnormalized) -----")
display(oxide_results)

oxide_total_wt_percent = oxide_results["oxide_wt_percent"].sum()

print("\n----- Oxide total -----")
print(f"{oxide_total_wt_percent:.2f} wt%")

# save the oxide table beside the elemental XPP results.
oxide_results_csv = project_dir / "C1_xpp_oxide_results.csv"
oxide_results.to_csv(oxide_results_csv, index=False)

print(f"\nWrote oxide results to: {oxide_results_csv}")

----- Oxide-equivalent results (unnormalized) -----


,oxide,source_element,element_mass_fraction,element_wt_percent,element_to_oxide_factor,oxide_wt_percent,oxide_mass_fraction
0,Na2O,Na,0.072156,7.215605,1.347956,9.726315,0.097263
1,Al2O3,Al,0.078527,7.852688,1.889426,14.837075,0.148371
2,MgO,Mg,0.039126,3.912610,1.658260,6.488122,0.064881
3,SiO2,Si,0.269831,26.983089,2.139286,57.724556,0.577246
4,CaO,Ca,0.072397,7.239739,1.399197,10.129818,0.101298
5,FeO,Fe,0.025392,2.539235,1.286489,3.266699,0.032667



----- Oxide total -----
102.17 wt%

Wrote oxide results to: /Users/thomaslam/Documents/NeXL_XPP/C1_xpp_oxide_results.csv


In [29]:
# Applying Corrected C3 mask to original data cube prior to NMF 
cube = s.rebin((1024/8, 768/8, 2665)).data

# Corrected component 3
mask = io.imread("output/nmf_component_3_mask_clean.png")
keep = mask > 128

cube3 = cube.reshape(-1, cube.shape[-1])
keep3 = keep.reshape(-1)

sum_spec_array = cube3[keep3].sum(axis=0)

# energy axis from original signal
x = s.axes_manager.signal_axes[0].axis

plt.figure(figsize=(8, 4))
plt.plot(x, sum_spec_array, lw=1)
plt.xlim(0.2, 7)
plt.xlabel(f"{s.axes_manager.signal_axes[0].name} ({s.axes_manager.signal_axes[0].units})")
plt.ylabel("Counts")
plt.title("C_Component 3 summed spectrum")

plt.tight_layout()
plt.show()

In [30]:
# C_Quantification for Corrected Component 3

# Deconvolution and background removal for Corrected Component 3
# Energy axis from the HyperSpy signal object
energy_keV = np.asarray(
    s.axes_manager.signal_axes[0].axis,
    dtype=float
)

# Counts from the NumPy summed-spectrum array
counts = np.asarray(
    sum_spec_array,
    dtype=float
).squeeze()

# If the energy axis is in eV rather than keV, convert it.
# Comment this out if energy_keV is already in keV.
if np.nanmax(energy_keV) > 100:
    energy_keV = energy_keV / 1000.0

if energy_keV.ndim != 1:
    raise ValueError(
        f"Energy axis must be 1D; got shape {energy_keV.shape}"
    )

if counts.ndim != 1:
    raise ValueError(
        f"Counts must be 1D; got shape {counts.shape}"
    )

if energy_keV.size != counts.size:
    raise ValueError(
        f"Energy axis has {energy_keV.size} channels, "
        f"but counts has {counts.size} values."
    )

print("Type of sum_spec_array:", type(sum_spec_array))
print("Energy-axis shape:", energy_keV.shape)
print("Counts shape:", counts.shape)
print(
    "Energy range:",
    energy_keV.min(),
    "to",
    energy_keV.max(),
    "keV"
)


# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "O_Ka": 0.525
}


# ============================================================
# Fit O peaks plus local continuum
# ============================================================

def fit_low_energy_eds(
    energy_keV,
    counts,
    e_min=0.3,
    e_max=0.7,
    fwhm_eV=100.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Fit O Kα,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_low_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "O_Ka": "tab:blue"
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "O EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_low_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=0.3,
    e_max=0.7,
    fwhm_eV=100.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
O_net = areas.loc[
    areas["line"].eq("O_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("O_net:", O_net)



# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit ((pop up gui)
fig = plot_low_energy_fit(
    result
)

plt.show()

Type of sum_spec_array: <class 'numpy.ndarray'>
Energy-axis shape: (2665,)
Counts shape: (2665,)
Energy range: -0.96 to 25.68 keV
[[Model]]
    (Model(linear, prefix='bg_') + Model(gaussian, prefix='O_Ka_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 75
    # data points      = 40
    # variables        = 4
    chi-square         = 3327.69096
    reduced chi-square = 92.4358600
    Akaike info crit   = 184.846179
    Bayesian info crit = 191.601697
    R-squared          = 0.98632921
[[Variables]]
    bg_slope:       -201.795016 +/- 181.441861 (89.91%) (init = -726.9231)
    bg_intercept:    263.894324 +/- 92.3706227 (35.00%) (init = 670.0769)
    O_Ka_amplitude:  911.080774 +/- 29.7580217 (3.27%) (init = 894.7569)
    O_Ka_center:     0.525 (fixed)
    O_Ka_sigma:      0.02547965 +/- 6.6813e-04 (2.62%) (init = 0.04246609)
    O_Ka_fwhm:       0.06000000 +/- 0.00157331 (2.62%) == '2.3548200*O_Ka_sigma'
    O_Ka_height:     14265.0545 +/- 574.0145

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,O_Ka,911.080774,91108.077405,29.758022,0.525,25.479654,60.0


O_net: 911.0807740529552
Continuum-subtracted spectrum generated: (40,)


In [31]:
# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "Na_Ka": 1.04098,
    "Mg_Ka": 1.25400,
    "Al_Ka": 1.48670,
    "Si_Ka": 1.73998,
}


# ============================================================
# Fit Na, Mg, Al, Si peaks plus local continuum
# ============================================================

def fit_low_energy_eds(
    energy_keV,
    counts,
    e_min=0.90,
    e_max=1.90,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Simultaneously fit Na Kα, Mg Kα, Al Kα, Si Kα,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_low_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "Na_Ka": "tab:blue",
        "Mg_Ka": "tab:red",
        "Al_Ka": "tab:green",
        "Si_Ka": "tab:orange",
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "Na–Mg–Al–Si simultaneous EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_low_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=0.90,
    e_max=1.90,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
Na_net = areas.loc[
    areas["line"].eq("Na_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Mg_net = areas.loc[
    areas["line"].eq("Mg_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Al_net = areas.loc[
    areas["line"].eq("Al_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Si_net = areas.loc[
    areas["line"].eq("Si_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("Si_net:", Si_net)
print("Mg_net:", Mg_net)
print("Al_net:", Al_net)
print("Si_net:", Si_net)


# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit ((pop up gui)
fig = plot_low_energy_fit(
    result
)

plt.show()

[[Model]]
    ((((Model(linear, prefix='bg_') + Model(gaussian, prefix='Na_Ka_')) + Model(gaussian, prefix='Mg_Ka_')) + Model(gaussian, prefix='Al_Ka_')) + Model(gaussian, prefix='Si_Ka_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 301
    # data points      = 101
    # variables        = 10
    chi-square         = 2385.91458
    reduced chi-square = 26.2188415
    Akaike info crit   = 339.383946
    Bayesian info crit = 365.535151
    R-squared          = 0.99806933
[[Variables]]
    bg_slope:         336.498431 +/- 63.7076942 (18.93%) (init = 552)
    bg_intercept:    -77.0666019 +/- 93.7130941 (121.60%) (init = 0)
    Na_Ka_amplitude:  470.677963 +/- 13.3570003 (2.84%) (init = 417.1552)
    Na_Ka_center:     1.04098 (fixed)
    Na_Ka_sigma:      0.03439753 +/- 8.4032e-04 (2.44%) (init = 0.05732922)
    Mg_Ka_amplitude:  10.8198482 +/- 9.76380213 (90.24%) (init = 1)
    Mg_Ka_center:     1.254 (fixed)
    Mg_Ka_sigma:      0.05847323 +/- 0.04

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,Na_Ka,470.677963,47067.796287,13.357000,1.04098,34.397534,81.000000
1,Mg_Ka,10.819848,1081.984824,9.763802,1.25400,58.473231,137.693934
2,Al_Ka,753.566335,75356.633461,15.716096,1.48670,34.397534,81.000000
3,Si_Ka,1858.301452,185830.145192,23.854390,1.73998,34.613348,81.508204


Si_net: 1858.3014519155095
Mg_net: 10.819848242269215
Al_net: 753.5663346144859
Si_net: 1858.3014519155095
Continuum-subtracted spectrum generated: (101,)


In [32]:
# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "K_Ka": 3.312,
    "K_Kb": 3.589,
    "Ca_Ka": 3.69
}


# ============================================================
# Fit K Ka and Kb peaks plus local continuum
# ============================================================

def fit_energy_eds(
    energy_keV,
    counts,
    e_min=3.0,
    e_max=4.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Simultaneously fit Ca Kα, Ca Kb,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "K_Ka": "tab:blue",
        "K_Kb": "tab:blue",
        "Ca_Ka": "tab:red"
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "simultaneous EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=3.0,
    e_max=4.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
K_net = areas.loc[
    areas["line"].eq("K_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("K_net:", K_net)

Ca_net = areas.loc[
    areas["line"].eq("Ca_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("Ca_net:", Ca_net)


# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit (pop up gui)
fig = plot_energy_fit(
    result
)

plt.show()

[[Model]]
    (((Model(linear, prefix='bg_') + Model(gaussian, prefix='K_Ka_')) + Model(gaussian, prefix='K_Kb_')) + Model(gaussian, prefix='Ca_Ka_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 108
    # data points      = 151
    # variables        = 8
    chi-square         = 172.786822
    reduced chi-square = 1.20829945
    Akaike info crit   = 36.3515918
    Bayesian info crit = 60.4898304
    R-squared          = 0.91550446
[[Variables]]
    bg_slope:        -54.2838971 +/- 3.01748279 (5.56%) (init = -47)
    bg_intercept:     349.765302 +/- 12.1552827 (3.48%) (init = 316.5)
    K_Ka_amplitude:   15.0869492 +/- 1.00135587 (6.64%) (init = 13.7666)
    K_Ka_center:      3.312 (fixed)
    K_Ka_sigma:       0.04420056 +/- 0.00297017 (6.72%) (init = 0.05732922)
    K_Kb_amplitude:   4.97242849 +/- 1.78226745 (35.84%) (init = 3.0894)
    K_Kb_center:      3.589 (fixed)
    K_Kb_sigma:       0.10319260 +/- 0.03152877 (30.55%) (init = 0.05732922)
 

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,K_Ka,15.086949,1508.694923,1.001356,3.312,44.200555,104.084352
1,K_Kb,4.972428,497.242849,1.782267,3.589,103.192601,243.000000
2,Ca_Ka,2.419716,241.971558,0.851178,3.690,34.397534,81.000000


K_net: 15.086949230634046
Ca_net: 2.4197155791428826
Continuum-subtracted spectrum generated: (151,)


In [33]:
# k-ratios corrected for the area fraction of the phase being quantified
std_df = pd.read_csv("15kV_Stds_Lib.csv")
std_row = std_df.iloc[0]

# Enter area fraction of the phase of interest (e.g., from the Otsu/multi-Otsu mask)
PHASE_AFRAC = 0.0554         # or: PHASE_AFRAC = C_C3_AFrac

if not np.isfinite(PHASE_AFRAC) or not (0 < PHASE_AFRAC <= 1):
    raise ValueError(f"Invalid phase area fraction ({PHASE_AFRAC}); must be in (0, 1].")

# Standard net counts (zero-based column positions in 15kV_Stds_Lib.csv)
std_net_counts = {
    "O":  pd.to_numeric(std_row.iloc[12], errors="raise"),
    "Na": pd.to_numeric(std_row.iloc[1],  errors="raise"),
    "Mg": pd.to_numeric(std_row.iloc[2],  errors="raise"),
    "Al": pd.to_numeric(std_row.iloc[3],  errors="raise"),
    "Si": pd.to_numeric(std_row.iloc[4],  errors="raise"),
    "K":  pd.to_numeric(std_row.iloc[7],  errors="raise"), 
    "Ca": pd.to_numeric(std_row.iloc[8],  errors="raise")
}

unknown_net_counts = {
    "O":  O_net,
    "Na": Na_net,
    "Mg": Mg_net,
    "Al": Al_net,
    "Si": Si_net,
    "K":  K_net,
    "Ca": Ca_net
}

# Every element must have both a standard and an unknown value
if set(std_net_counts) != set(unknown_net_counts):
    raise ValueError(
        f"Element mismatch: standards {sorted(std_net_counts)} vs "
        f"unknowns {sorted(unknown_net_counts)}"
    )

# Validate inputs
for element, i_std in std_net_counts.items():
    if not np.isfinite(i_std) or i_std <= 0:
        raise ValueError(f"{element}: invalid standard net count ({i_std}).")
    i_unknown = float(unknown_net_counts[element])
    if not np.isfinite(i_unknown) or i_unknown < 0:
        raise ValueError(f"{element}: invalid unknown net count ({i_unknown}).")

# k = I_unknown / (I_standard * area fraction)
kr_df = pd.DataFrame(
    {
        "element": list(std_net_counts.keys()),
        "k_ratio": [
            float(unknown_net_counts[el]) / (float(std_net_counts[el]) * PHASE_AFRAC)
            for el in std_net_counts
        ],
    }
)

print(f"Measured k-ratios (phase area fraction = {PHASE_AFRAC:.4f})")
display(kr_df.style.format({"k_ratio": "{:.6f}"}))

# Export for NeXL
input_csv = Path("input_kratios.csv").resolve()
kr_df[["element", "k_ratio"]].to_csv(input_csv, index=False, float_format="%.6f")
print(f"Saved: {input_csv}")

Measured k-ratios (phase area fraction = 0.0554)


,element,k_ratio
0,O,1.432940
1,Na,1.097756
2,Mg,0.002529
3,Al,1.223128
4,Si,0.961306
5,K,0.030034
6,Ca,0.001913


Saved: /Users/thomaslam/Downloads/input_kratios.csv


In [34]:
from pathlib import Path

workdir = Path("/Users/thomaslam/Documents/NeXL_XPP")
workdir.mkdir(exist_ok=True)

input_csv = workdir / "input_kratios.csv"
kr_df.to_csv(input_csv, index=False)

print(f"Saved NeXL input: {input_csv}")
print(kr_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

Saved NeXL input: /Users/thomaslam/Documents/NeXL_XPP/input_kratios.csv
element  k_ratio
      O 1.432940
     Na 1.097756
     Mg 0.002529
     Al 1.223128
     Si 0.961306
      K 0.030034
     Ca 0.001913


In [35]:
from pathlib import Path
import subprocess
import textwrap
import pandas as pd

# ================================================================
# 1. Python-controlled inputs
# ================================================================
BEAM_ENERGY_KEV = 15.0
TAKE_OFF_ANGLE_DEG = 35.0

input_csv = Path(input_csv).resolve()
standards_csv = Path("/Users/thomaslam/Documents/NeXL_XPP/Standards_df.csv")

project_dir = Path("/Users/thomaslam/Documents/NeXL_XPP")
project_dir.mkdir(parents=True, exist_ok=True)

results_csv = project_dir / "xpp_results.csv"
julia_script = project_dir / "run_xpp_from_python.jl"

# ================================================================
# 2. Validate Python inputs and write k-ratio CSV
# ================================================================
if not input_csv.is_file():
    raise FileNotFoundError(f"Python cannot find:\n{input_csv}")

if not standards_csv.is_file():
    raise FileNotFoundError(f"Python cannot find:\n{standards_csv}")

kr_df = kr_df.copy()
kr_df.columns = kr_df.columns.astype(str).str.strip()

required_columns = {"element", "k_ratio"}
if not required_columns.issubset(set(kr_df.columns)):
    raise ValueError(
        f"kr_df must contain {required_columns}; found {list(kr_df.columns)}"
    )

kr_df["element"] = kr_df["element"].astype(str).str.strip()
kr_df["k_ratio"] = pd.to_numeric(kr_df["k_ratio"], errors="raise")

if kr_df["element"].duplicated().any():
    duplicates = kr_df.loc[
        kr_df["element"].duplicated(keep=False), "element"
    ].tolist()
    raise ValueError(
        "Each element must appear only once in kr_df. "
        f"Duplicate entries: {duplicates}"
    )

if (kr_df["k_ratio"] < 0).any():
    raise ValueError("All k_ratio values must be non-negative.")

print("Python input k-ratios:")
display(kr_df[["element", "k_ratio"]])

kr_df[["element", "k_ratio"]].to_csv(input_csv, index=False)

# ================================================================
# 3. Julia backend written by Python
# ================================================================
julia_code = r'''
using CSV
using DataFrames
using NeXLMatrixCorrection

input_kratios_csv = ARGS[1]
standards_csv = ARGS[2]
results_csv = ARGS[3]
beam_energy_kev = parse(Float64, ARGS[4])
take_off_angle_deg = parse(Float64, ARGS[5])

kr_df = CSV.read(input_kratios_csv, DataFrame)
standards_df = CSV.read(standards_csv, DataFrame)

required_columns = ["element", "k_ratio"]
found_columns = String.(names(kr_df))
missing_columns = setdiff(required_columns, found_columns)

isempty(missing_columns) || error(
    "input_kratios.csv missing columns: $(missing_columns). " *
    "Found: $(found_columns)"
)

println(
    "Quantification conditions from Python: E0 = ", beam_energy_kev,
    " keV; TOA = ", take_off_angle_deg, " degrees"
)

unknown_properties = Dict(
    :BeamEnergy => beam_energy_kev * 1.0e3,
    :TakeOffAngle => deg2rad(take_off_angle_deg)
)

standard_properties = Dict(
    :BeamEnergy => beam_energy_kev * 1.0e3,
    :TakeOffAngle => deg2rad(take_off_angle_deg)
)

const ATOMIC_WEIGHT = Dict(
    "O"  => 15.999,
    "Na" => 22.990,
    "Mg" => 24.305,
    "Al" => 26.982,
    "Si" => 28.086,
    "K"  => 39.098,
    "Ca" => 40.078,
    "Ti" => 47.867,
    "Mn" => 54.938,
    "Fe" => 55.845
)

const OXIDE_DEFINITIONS = Dict(
    :SiO2  => ("Si", 1, 2),
    :TiO2  => ("Ti", 1, 2),
    :Al2O3 => ("Al", 2, 3),
    :FeO   => ("Fe", 1, 1),
    :Fe2O3 => ("Fe", 2, 3),
    :MnO   => ("Mn", 1, 1),
    :MgO   => ("Mg", 1, 1),
    :CaO   => ("Ca", 1, 1),
    :Na2O  => ("Na", 2, 1),
    :K2O   => ("K", 2, 1)
)

function elemental_fractions_from_oxides(row)
    elemental_wtpercent = Dict{String, Float64}()

    for (oxide, (element, n_element, n_oxygen)) in OXIDE_DEFINITIONS
        oxide in propertynames(row) || continue

        wtpercent = row[oxide]

        if ismissing(wtpercent) || Float64(wtpercent) <= 0.0
            continue
        end

        wtpercent = Float64(wtpercent)

        molecular_weight =
            n_element * ATOMIC_WEIGHT[element] +
            n_oxygen * ATOMIC_WEIGHT["O"]

        elemental_wtpercent[element] =
            get(elemental_wtpercent, element, 0.0) +
            wtpercent * n_element * ATOMIC_WEIGHT[element] / molecular_weight

        elemental_wtpercent["O"] =
            get(elemental_wtpercent, "O", 0.0) +
            wtpercent * n_oxygen * ATOMIC_WEIGHT["O"] / molecular_weight
    end

    total = sum(values(elemental_wtpercent))

    total > 0.0 || error(
        "No positive oxide wt% values found for standard: " *
        String(row.standard_id)
    )

    return Dict(
        element => wtpercent / total
        for (element, wtpercent) in elemental_wtpercent
    )
end

function get_standard_fractions(standard_id)
    hits = filter(
        row -> strip(String(row.standard_id)) == standard_id,
        standards_df
    )

    nrow(hits) == 1 || error(
        "Expected exactly one standard row for \"" * standard_id *
        "\"; found $(nrow(hits))."
    )

    return elemental_fractions_from_oxides(hits[1, :])
end

function make_material(standard_id)
    fractions = get_standard_fractions(standard_id)

    composition = join(
        [
            string(value, "*", element)
            for (element, value) in sort(collect(fractions); by=first)
        ],
        "+"
    )

    return parse(Material, composition; name=standard_id)
end

standards = Dict(
    "O"  => make_material("orthoclase_spi_mci"),
    "Na" => make_material("albite_spi_mci"),
    "Mg" => make_material("MgO_spi_mci"),
    "Al" => make_material("orthoclase_spi_mci"),
    "Si" => make_material("orthoclase_spi_mci"),
    "K" => make_material("orthoclase_spi_mci"),
    "Ca" => make_material("wollastonite_spi_mci"),
    "Mn" => make_material("rhodonite_spi_mci"),
    "Fe" => make_material("hematite_spi_mci")
)

lines = Dict(
    "O"  => n"O K-L3",
    "Na" => n"Na K-L3",
    "Mg" => n"Mg K-L3",
    "Al" => n"Al K-L3",
    "Si" => n"Si K-L3",
    "K" => n"K K-L3",
    "Ca" => n"Ca K-L3",
    "Mn" => n"Mn K-L3",
    "Fe" => n"Fe K-L3"
)

krs = KRatio[]

for row in eachrow(kr_df)
    element = strip(String(row.element))

    haskey(standards, element) || error(
        "No standard mapping for element: \"$element\""
    )

    k = Float64(row.k_ratio)
    k >= 0.0 || error("Negative k-ratio for $element")

    # Uses a 1% relative k-ratio uncertainty, with a small positive floor.
    sigma_k = max(0.01 * max(k, 1.0e-6), 1.0e-6)

    push!(
        krs,
        KRatio(
            [lines[element]],
            unknown_properties,
            standard_properties,
            standards[element],
            uv(k, sigma_k)
        )
    )
end

println("K-ratios received from Python:")
foreach(println, krs)

# NeXL quantification has completed successfully in this environment.
result = quantify(label("python_xpp"), krs)

# ----------------------------------------------------------------
# Export directly from IterationResult; do NOT call asa(DataFrame,...)
# because the installed NeXL DataFrames extension references _ZAF,
# which is undefined in that package version.
# ----------------------------------------------------------------

# `result.comp` is the final quantified Material. Build the CSV in
# input-element order, so Python receives a predictable tidy table.
result_elements = String[]
mass_fractions = Float64[]

for row in eachrow(kr_df)
    element_name = strip(String(row.element))

    # `parse(Element, ...)` creates the element key used by Material.
    element = parse(Element, element_name)

    # The Material is indexed by Element and returns the elemental
    # mass fraction. This bypasses NeXL's broken DataFrames extension.
    mass_fraction = Float64(result.comp[element])

    push!(result_elements, element_name)
    push!(mass_fractions, mass_fraction)
end

out_df = DataFrame(
    label = fill(string(result.label), length(result_elements)),
    converged = fill(Bool(result.converged), length(result_elements)),
    iterations = fill(Int(result.iterations), length(result_elements)),
    beam_energy_kev = fill(beam_energy_kev, length(result_elements)),
    take_off_angle_deg = fill(take_off_angle_deg, length(result_elements)),
    element = result_elements,
    mass_fraction = mass_fractions,
    wt_percent = 100.0 .* mass_fractions
)

CSV.write(results_csv, out_df)

println("Final quantification result: ", result)
println("Wrote results to: ", results_csv)
'''

julia_script.write_text(
    textwrap.dedent(julia_code),
    encoding="utf-8"
)

# ================================================================
# 4. Python launches Julia and reads results back
# ================================================================
run = subprocess.run(
    [
        "julia",
        str(julia_script),
        str(input_csv),
        str(standards_csv),
        str(results_csv),
        str(BEAM_ENERGY_KEV),
        str(TAKE_OFF_ANGLE_DEG),
    ],
    text=True,
    capture_output=True,
)

print("----- Julia stdout -----")
print(run.stdout)

if run.returncode != 0:
    print("----- Julia stderr -----")
    print(run.stderr)
    raise RuntimeError("Julia/NeXL quantification did not complete.")

if not results_csv.is_file():
    raise FileNotFoundError(
        f"Julia did not create the results output:\n{results_csv}"
    )

# ----- Load the Julia/NeXL results CSV back into Python -----
xpp_results = pd.read_csv(results_csv)

print("----- Results returned to Python -----")
display(xpp_results)

print("\n----- Quantified elemental total -----")
print(f"{xpp_results['mass_fraction'].sum():.6f}")

Python input k-ratios:


,element,k_ratio
0,O,1.432940
1,Na,1.097756
2,Mg,0.002529
3,Al,1.223128
4,Si,0.961306
5,K,0.030034
6,Ca,0.001913


----- Julia stdout -----
Quantification conditions from Python: E0 = 15.0 keV; TOA = 35.0 degrees
K-ratios received from Python:
k[O K-L3, orthoclase_spi_mci] = 1.433 ± 0.014
k[Na K-L3, albite_spi_mci] = 1.098 ± 0.011
k[Mg K-L3, MgO_spi_mci] = 2.529e-03 ± 2.5e-05
k[Al K-L3, orthoclase_spi_mci] = 1.223 ± 0.012
k[Si K-L3, orthoclase_spi_mci] = 0.9613 ± 0.0096
k[K K-L3, orthoclase_spi_mci] = 0.0300 ± 0.0003
k[Ca K-L3, wollastonite_spi_mci] = 1.913e-03 ± 1.9e-05
Final quantification result: Converged to python_xpp[O=0.4711,Na=0.0934,Mg=0.0019,Al=0.1136,Si=0.3091,K=0.0040,Ca=0.0007] in 10 steps.
Wrote results to: /Users/thomaslam/Documents/NeXL_XPP/xpp_results.csv

----- Results returned to Python -----


,label,converged,iterations,beam_energy_kev,take_off_angle_deg,element,mass_fraction,wt_percent
0,python_xpp,True,10,15.0,35.0,O,0.471057,47.105745
1,python_xpp,True,10,15.0,35.0,Na,0.093420,9.342013
2,python_xpp,True,10,15.0,35.0,Mg,0.001859,0.185877
3,python_xpp,True,10,15.0,35.0,Al,0.113650,11.364961
4,python_xpp,True,10,15.0,35.0,Si,0.309128,30.912830
5,python_xpp,True,10,15.0,35.0,K,0.003951,0.395096
6,python_xpp,True,10,15.0,35.0,Ca,0.000692,0.069202



----- Quantified elemental total -----
0.993757


In [36]:
# ================================================================
# 5. Convert quantified elemental mass fractions to oxide equivalents
# ================================================================

# Select the reporting convention for iron:
# "FeO"   = report all quantified Fe as ferrous iron
# "Fe2O3" = report all quantified Fe as ferric iron
IRON_OXIDE = "FeO"

# Atomic weights in g/mol
ATOMIC_WEIGHT = {
    "O": 15.999,
    "Na": 22.990,
    "Mg": 24.305,
    "Al": 26.982,
    "Si": 28.086,
    "Ca": 40.078,
    "K": 39.098,
}

# Oxide stoichiometry:
# oxide_name: (cation_symbol, number_of_cations, number_of_oxygens)
OXIDE_DEFINITIONS = {
    "Na2O": ("Na", 2, 1),
    "Al2O3": ("Al", 2, 3),
    "MgO": ("Mg", 1, 1),
    "SiO2": ("Si", 1, 2),
    "K2O": ("K", 2, 1),
    "CaO": ("Ca", 1, 1),
}

# Element wt% -> oxide wt% factor:
# factor = molar_mass(oxide) / mass_of_cations_in_one_formula_unit
oxide_rows = []

for oxide, (element, n_cation, n_oxygen) in OXIDE_DEFINITIONS.items():
    match = xpp_results.loc[
        xpp_results["element"].astype(str).str.strip() == element
    ]

    if match.empty:
        print(f"Warning: no quantified {element} row; skipping {oxide}.")
        continue

    elemental_mass_fraction = float(match["mass_fraction"].iloc[0])
    elemental_wt_percent = 100.0 * elemental_mass_fraction

    oxide_molar_mass = (
        n_cation * ATOMIC_WEIGHT[element]
        + n_oxygen * ATOMIC_WEIGHT["O"]
    )

    cation_mass_per_formula_unit = n_cation * ATOMIC_WEIGHT[element]

    element_to_oxide_factor = (
        oxide_molar_mass / cation_mass_per_formula_unit
    )

    oxide_wt_percent = elemental_wt_percent * element_to_oxide_factor

    oxide_rows.append(
        {
            "oxide": oxide,
            "source_element": element,
            "element_mass_fraction": elemental_mass_fraction,
            "element_wt_percent": elemental_wt_percent,
            "element_to_oxide_factor": element_to_oxide_factor,
            "oxide_wt_percent": oxide_wt_percent,
            "oxide_mass_fraction": oxide_wt_percent / 100.0,
        }
    )

oxide_results = pd.DataFrame(oxide_rows)

print("----- Oxide-equivalent results (unnormalized) -----")
display(oxide_results)

oxide_total_wt_percent = oxide_results["oxide_wt_percent"].sum()

print("\n----- Oxide total -----")
print(f"{oxide_total_wt_percent:.2f} wt%")

# save the oxide table beside the elemental XPP results.
oxide_results_csv = project_dir / "C_C3_xpp_oxide_results.csv"
oxide_results.to_csv(oxide_results_csv, index=False)

print(f"\nWrote oxide results to: {oxide_results_csv}")

----- Oxide-equivalent results (unnormalized) -----


,oxide,source_element,element_mass_fraction,element_wt_percent,element_to_oxide_factor,oxide_wt_percent,oxide_mass_fraction
0,Na2O,Na,0.093420,9.342013,1.347956,12.592619,0.125926
1,Al2O3,Al,0.113650,11.364961,1.889426,21.473256,0.214733
2,MgO,Mg,0.001859,0.185877,1.658260,0.308233,0.003082
3,SiO2,Si,0.309128,30.912830,2.139286,66.131400,0.661314
4,K2O,K,0.003951,0.395096,1.204601,0.475933,0.004759
5,CaO,Ca,0.000692,0.069202,1.399197,0.096827,0.000968



----- Oxide total -----
101.08 wt%

Wrote oxide results to: /Users/thomaslam/Documents/NeXL_XPP/C_C3_xpp_oxide_results.csv


In [37]:
#Image proceesing to account component 3, 2, 4 in component 0, to yield C_C0
from PIL import Image, ImageOps

outdir = Path("output")

# Load masks as grayscale images
c3 = Image.open(outdir / "nmf_component_3_mask_clean.png").convert("L")
c2 = Image.open(outdir / "nmf_component_2_mask.png").convert("L")
c4 = Image.open(outdir / "nmf_component_4_mask.png").convert("L")
c0 = Image.open(outdir / "nmf_component_0_mask.png").convert("L")

# Check that all masks have matching dimensions
if not (c0.size == c2.size == c3.size == c4.size):
    raise ValueError(
        "All masks must have the same dimensions. "
        f"C0: {c0.size}, C2: {c2.size}, "
        f"C3: {c3.size}, C4: {c4.size}"
    )

# Invert the masks to be excluded from C0
c3_inv = ImageOps.invert(c3)
c2_inv = ImageOps.invert(c2)
c4_inv = ImageOps.invert(c4)

c3_inv.save(outdir / "c3_inv.png")
c2_inv.save(outdir / "c2_inv.png")
c4_inv.save(outdir / "c4_inv.png")

# Convert to arrays
a3 = np.array(c3_inv)
a2 = np.array(c2_inv)
a4 = np.array(c4_inv)  
a0 = np.array(c0)

# Pixelwise minimum across all four masks
combined = np.minimum.reduce([a0, a2, a4, a3])

# Save combined mask
result = Image.fromarray(combined.astype(np.uint8))
result.save(outdir / "C_C0_combined.png")

In [38]:
# Corrected component 0 area fraction calc
img_rgba = io.imread('output/C_C0_combined.png')

# Convert RGBA to grayscale (take just the first channel or luminance)
if len(img_rgba.shape) == 3 and img_rgba.shape[2] == 4:
    # Take first channel (grayscale saved as RGBA)
    img = img_rgba[:, :, 0].astype(np.float32)
    print(f"Converted RGBA to grayscale. Shape: {img.shape}")
else:
    img = img_rgba.astype(np.float32)

# Apply Otsu thresholding
threshold = threshold_otsu(img)
binary = img >= threshold
# Get area fraction and save as variable C_C3_AFrac, where C_C3 is the Corrected Component 3
C_C0_AFrac = binary.sum() / binary.size

print(f"  C_C0_AFrac = {C_C0_AFrac:.2%}")
print(f"  Foreground pixels: {binary.sum()}")
print(f"  Background pixels: {(~binary).sum()}")
print(f"  Total pixels: {binary.size}")

  C_C0_AFrac = 83.01%
  Foreground pixels: 10200
  Background pixels: 2088
  Total pixels: 12288


In [39]:
# Applying Corrected C0 mask to original data cube prior to NMF 
cube = s.rebin((1024/8, 768/8, 2665)).data

# Corrected component 3
mask = io.imread("output/C_C0_combined.png")
keep = mask > 128

cube3 = cube.reshape(-1, cube.shape[-1])
keep3 = keep.reshape(-1)

sum_spec_array = cube3[keep3].sum(axis=0)

# energy axis from original signal
x = s.axes_manager.signal_axes[0].axis

plt.figure(figsize=(8, 4))
plt.plot(x, sum_spec_array, lw=1)
plt.xlim(0.2, 7)
plt.xlabel(f"{s.axes_manager.signal_axes[0].name} ({s.axes_manager.signal_axes[0].units})")
plt.ylabel("Counts")
plt.title("C_Component 0 summed spectrum")

plt.tight_layout()
plt.show()

In [40]:
# Deconvolution and background removal for Corrected Component 0
# Energy axis from the HyperSpy signal object
energy_keV = np.asarray(
    s.axes_manager.signal_axes[0].axis,
    dtype=float
)

# Counts from the NumPy summed-spectrum array
counts = np.asarray(
    sum_spec_array,
    dtype=float
).squeeze()

# If the energy axis is in eV rather than keV, convert it.
# Comment this out if energy_keV is already in keV.
if np.nanmax(energy_keV) > 100:
    energy_keV = energy_keV / 1000.0

if energy_keV.ndim != 1:
    raise ValueError(
        f"Energy axis must be 1D; got shape {energy_keV.shape}"
    )

if counts.ndim != 1:
    raise ValueError(
        f"Counts must be 1D; got shape {counts.shape}"
    )

if energy_keV.size != counts.size:
    raise ValueError(
        f"Energy axis has {energy_keV.size} channels, "
        f"but counts has {counts.size} values."
    )

print("Type of sum_spec_array:", type(sum_spec_array))
print("Energy-axis shape:", energy_keV.shape)
print("Counts shape:", counts.shape)
print(
    "Energy range:",
    energy_keV.min(),
    "to",
    energy_keV.max(),
    "keV"
)


# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "O_Ka": 0.525
}


# ============================================================
# Fit O peaks plus local continuum
# ============================================================

def fit_low_energy_eds(
    energy_keV,
    counts,
    e_min=0.3,
    e_max=0.7,
    fwhm_eV=100.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Fit O Kα,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_low_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "O_Ka": "tab:blue"
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "O EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_low_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=0.3,
    e_max=0.7,
    fwhm_eV=100.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
O_net = areas.loc[
    areas["line"].eq("O_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("O_net:", O_net)



# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit ((pop up gui)
fig = plot_low_energy_fit(
    result
)

plt.show()

Type of sum_spec_array: <class 'numpy.ndarray'>
Energy-axis shape: (2665,)
Counts shape: (2665,)
Energy range: -0.96 to 25.68 keV
[[Model]]
    (Model(linear, prefix='bg_') + Model(gaussian, prefix='O_Ka_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 75
    # data points      = 40
    # variables        = 4
    chi-square         = 49152.6726
    reduced chi-square = 1365.35202
    Akaike info crit   = 292.552282
    Bayesian info crit = 299.307800
    R-squared          = 0.98514643
[[Variables]]
    bg_slope:       -2328.18439 +/- 2816.08744 (120.96%) (init = -9006.41)
    bg_intercept:    3765.16854 +/- 1413.37640 (37.54%) (init = 9116.423)
    O_Ka_amplitude:  13735.1864 +/- 444.913346 (3.24%) (init = 13521.36)
    O_Ka_center:     0.525 (fixed)
    O_Ka_sigma:      0.02547965 +/- 6.6438e-04 (2.61%) (init = 0.04246609)
    O_Ka_fwhm:       0.06000000 +/- 0.00156449 (2.61%) == '2.3548200*O_Ka_sigma'
    O_Ka_height:     215055.775 +/- 8569.335

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,O_Ka,13735.18637,1.373519e+06,444.913346,0.525,25.479654,60.0


O_net: 13735.186369574794
Continuum-subtracted spectrum generated: (40,)


In [41]:
# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "Na_Ka": 1.04098,
    "Mg_Ka": 1.25400,
    "Al_Ka": 1.48670,
    "Si_Ka": 1.73998,
}


# ============================================================
# Fit Na, Mg, Al, Si peaks plus local continuum
# ============================================================

def fit_low_energy_eds(
    energy_keV,
    counts,
    e_min=0.90,
    e_max=1.90,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Simultaneously fit Na Kα, Mg Kα, Al Kα, Si Kα,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_low_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "Na_Ka": "tab:blue",
        "Mg_Ka": "tab:red",
        "Al_Ka": "tab:green",
        "Si_Ka": "tab:orange",
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "Na–Mg–Al–Si simultaneous EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_low_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=0.90,
    e_max=1.90,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
Na_net = areas.loc[
    areas["line"].eq("Na_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Mg_net = areas.loc[
    areas["line"].eq("Mg_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Al_net = areas.loc[
    areas["line"].eq("Al_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Si_net = areas.loc[
    areas["line"].eq("Si_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("Na_net:", Na_net)
print("Mg_net:", Mg_net)
print("Al_net:", Al_net)
print("Si_net:", Si_net)


# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit ((pop up gui)
fig = plot_low_energy_fit(
    result
)

plt.show()

[[Model]]
    ((((Model(linear, prefix='bg_') + Model(gaussian, prefix='Na_Ka_')) + Model(gaussian, prefix='Mg_Ka_')) + Model(gaussian, prefix='Al_Ka_')) + Model(gaussian, prefix='Si_Ka_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 143
    # data points      = 101
    # variables        = 10
    chi-square         = 35781.1059
    reduced chi-square = 393.198966
    Akaike info crit   = 612.875529
    Bayesian info crit = 639.026735
    R-squared          = 0.99730222
[[Variables]]
    bg_slope:         4848.71842 +/- 965.346366 (19.91%) (init = 6772.5)
    bg_intercept:    -1018.66938 +/- 1417.65911 (139.17%) (init = 693.25)
    Na_Ka_amplitude:  8537.16947 +/- 216.131770 (2.53%) (init = 7643.529)
    Na_Ka_center:     1.04098 (fixed)
    Na_Ka_sigma:      0.03439753 +/- 7.4753e-04 (2.17%) (init = 0.05732922)
    Mg_Ka_amplitude:  465.386785 +/- 122.381794 (26.30%) (init = 12.2205)
    Mg_Ka_center:     1.254 (fixed)
    Mg_Ka_sigma:      0.043

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,Na_Ka,8537.169468,8.537169e+05,216.131770,1.04098,34.397534,81.000000
1,Mg_Ka,465.386785,4.653868e+04,122.381794,1.25400,43.278619,101.913357
2,Al_Ka,12760.727335,1.276073e+06,247.363944,1.48670,34.397534,81.000000
3,Si_Ka,24672.343178,2.467234e+06,337.938416,1.73998,34.656415,81.609618


Na_net: 8537.169467727148
Mg_net: 465.3867846752378
Al_net: 12760.727335413741
Si_net: 24672.34317810709
Continuum-subtracted spectrum generated: (101,)


In [42]:
# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "Al_Si_sum": 3.23,
    "K_Ka": 3.312,
    "Si_Si_sum": 3.48,
    "K_Kb": 3.589,
    "Ca_Ka": 3.691,
    "Ca_Kb": 4.013
}


# ============================================================
# Fit Ca Ka and Kb peaks plus local continuum
# ============================================================

def fit_energy_eds(
    energy_keV,
    counts,
    e_min=3.0,
    e_max=4.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Simultaneously fit Ca Kα, Ca Kb,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "Al_Si_sum": "tab:purple",
        "K_Ka": "tab:blue",
        "Si_Si_sum": "tab:orange",
        "K_Kb": "tab:blue",
        "Ca_Ka": "tab:green",
        "Ca_Kb": "tab:green",
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "simultaneous EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=3.0,
    e_max=4.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
Ca_net = areas.loc[
    areas["line"].eq("Ca_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("Ca_net:", Ca_net)


# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit (pop up gui)
fig = plot_energy_fit(
    result
)

plt.show()

[[Model]]
    ((((((Model(linear, prefix='bg_') + Model(gaussian, prefix='Al_Si_sum_')) + Model(gaussian, prefix='K_Ka_')) + Model(gaussian, prefix='Si_Si_sum_')) + Model(gaussian, prefix='K_Kb_')) + Model(gaussian, prefix='Ca_Ka_')) + Model(gaussian, prefix='Ca_Kb_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 135
    # data points      = 151
    # variables        = 14
    chi-square         = 224.422104
    reduced chi-square = 1.63811755
    Akaike info crit   = 87.8335738
    Bayesian info crit = 130.075492
    R-squared          = 0.99358305
[[Variables]]
    bg_slope:            -759.443670 +/- 16.2502387 (2.14%) (init = -724.6667)
    bg_intercept:         5070.13953 +/- 66.9393220 (1.32%) (init = 4965)
    Al_Si_sum_amplitude:  35.1094092 +/- 5.84796149 (16.66%) (init = 28.77407)
    Al_Si_sum_center:     3.23 (fixed)
    Al_Si_sum_sigma:      0.03894248 +/- 0.00511199 (13.13%) (init = 0.05732922)
    K_Ka_amplitude:       1.4733e-09 +/-

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,Al_Si_sum,3.510941e+01,3.510941e+03,5.847961,3.230,38.942484,91.702540
1,K_Ka,1.473324e-09,1.473324e-07,9.012485,3.312,103.192601,243.000000
2,Si_Si_sum,2.500518e+01,2.500518e+03,8.152969,3.480,36.067451,84.932355
3,K_Kb,2.464257e-03,2.464257e-01,23.959020,3.589,103.185947,242.984332
4,Ca_Ka,3.511796e+02,3.511796e+04,10.990822,3.691,43.809517,103.163526
5,Ca_Kb,4.093826e+01,4.093826e+03,3.408194,4.013,43.687516,102.876237


Ca_net: 351.17956047177773
Continuum-subtracted spectrum generated: (151,)


In [43]:
# ============================================================
# Characteristic X-ray line energies
# ============================================================

LINE_ENERGIES_KEV = {
    "Mn_Ka": 5.893,
    "Mn_Kb": 6.491,
    "Fe_Ka": 6.397,
    "Fe_Kb": 7.058
}


# ============================================================
# Fit Mn Ka, Mn Kb, Fe Ka, Fe Kb peaks plus local continuum
# ============================================================

def fit_energy_eds(
    energy_keV,
    counts,
    e_min=5.5,
    e_max=7.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
):
    """
    Simultaneously fit Mn Ka, Mn Kb, Fe Kα, Fe Kb,
    plus a local linear or quadratic continuum.

    Parameters
    ----------
    energy_keV : array-like
        Energy axis in keV.

    counts : array-like
        Raw counts per channel.

    e_min, e_max : float
        Fitting limits in keV.

    fwhm_eV : float
        Initial detector FWHM estimate in eV.

    background : {"linear", "quadratic"}
        Continuum model.

    allow_energy_shift : bool
        If True, each peak center may move ±0.010 keV.

    Returns
    -------
    result : lmfit.model.ModelResult
        Complete fitting result.
    """

    energy_keV = np.asarray(
        energy_keV,
        dtype=float
    )

    counts = np.asarray(
        counts,
        dtype=float
    ).squeeze()

    valid = (
        np.isfinite(energy_keV)
        & np.isfinite(counts)
        & (energy_keV >= e_min)
        & (energy_keV <= e_max)
    )

    x = energy_keV[valid]
    y = counts[valid]

    if x.size < 30:
        raise ValueError(
            "Too few energy channels in the fitting range."
        )

    if np.any(np.diff(x) <= 0):
        raise ValueError(
            "Energy axis must be strictly increasing."
        )

    # Build the continuum model
    if background == "linear":
        model = LinearModel(prefix="bg_")
    elif background == "quadratic":
        model = QuadraticModel(prefix="bg_")
    else:
        raise ValueError(
            "background must be 'linear' or 'quadratic'."
        )

    # Add four Gaussian peak models
    for line in LINE_ENERGIES_KEV:
        model += GaussianModel(
            prefix=f"{line}_"
        )

    params = model.make_params()

    # Estimate an initial continuum from both ends
    n_end = max(
        3,
        min(10, x.size // 10)
    )

    left_level = np.median(y[:n_end])
    right_level = np.median(y[-n_end:])

    slope0 = (
        right_level - left_level
    ) / (
        x[-1] - x[0]
    )

    intercept0 = (
        left_level - slope0 * x[0]
    )

    params["bg_slope"].set(
        value=slope0
    )

    params["bg_intercept"].set(
        value=max(float(intercept0), 0.0)
    )

    # Detector width in keV
    sigma0 = (
        fwhm_eV / 1000.0
    ) / 2.35482

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    # Initial peak parameters
    for line, center in LINE_ENERGIES_KEV.items():

        local = np.abs(x - center) <= 0.06

        baseline = (
            intercept0
            + slope0 * x[local]
        )

        area0 = (
            np.sum(
                np.clip(
                    y[local] - baseline,
                    0,
                    None
                )
            )
            * channel_width_keV
        )

        area0 = max(
            float(area0),
            1.0
        )

        params[
            f"{line}_amplitude"
        ].set(
            value=area0,
            min=0.0
        )

        params[
            f"{line}_sigma"
        ].set(
            value=sigma0,
            min=0.60 * sigma0,
            max=1.80 * sigma0
        )

        if allow_energy_shift:
            params[
                f"{line}_center"
            ].set(
                value=center,
                min=center - 0.010,
                max=center + 0.010
            )
        else:
            params[
                f"{line}_center"
            ].set(
                value=center,
                vary=False
            )

    # Poisson-like weighting
    weights = 1.0 / np.sqrt(
        np.clip(y, 0, None) + 1.0
    )

    # Perform the fit
    result = model.fit(
        y,
        params=params,
        x=x,
        weights=weights,
        method="least_squares",
    )

    return result


# ============================================================
# Extract fitted peak areas
# ============================================================

def fitted_peak_areas(result):
    """
    Return deconvolved Gaussian peak areas.

    GaussianModel amplitude is the integrated peak area
    in count·keV when x is in keV.
    """

    x = result.userkws["x"]

    channel_width_keV = float(
        np.median(np.diff(x))
    )

    records = []

    for line in LINE_ENERGIES_KEV:

        area = result.params[
            f"{line}_amplitude"
        ]

        center = result.params[
            f"{line}_center"
        ]

        sigma = result.params[
            f"{line}_sigma"
        ]

        fwhm = result.params[
            f"{line}_fwhm"
        ]

        records.append(
            {
                "line": line,
                "net_peak_area_count_keV": area.value,
                "net_peak_counts": (
                    area.value
                    / channel_width_keV
                ),
                "area_standard_error": area.stderr,
                "fitted_center_keV": center.value,
                "sigma_eV": sigma.value * 1000.0,
                "fwhm_eV": fwhm.value * 1000.0,
            }
        )

    return pd.DataFrame(records)


# ============================================================
# Obtain continuum-subtracted spectrum
# ============================================================

def continuum_removed_spectrum(result):
    """
    Return the fitted energy axis, fitted continuum,
    and measured spectrum after continuum subtraction.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    continuum = components["bg_"]

    continuum_removed = y - continuum

    return x, y, continuum, continuum_removed


# ============================================================
# Plot complete fit and continuum-subtracted spectrum
# ============================================================

def plot_energy_fit(result):
    """
    Plot measured spectrum, total fit, fitted continuum,
    individual deconvolved peaks, and residuals.
    """

    x = result.userkws["x"]
    y = result.data

    components = result.eval_components(
        x=x
    )

    fit = result.best_fit
    background = components["bg_"]

    fig, (ax0, ax1) = plt.subplots(
        2,
        1,
        figsize=(10, 6.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1]
        },
    )

    ax0.step(
        x,
        y,
        where="mid",
        color="black",
        linewidth=1,
        label="Measured"
    )

    ax0.plot(
        x,
        fit,
        color="crimson",
        linewidth=2,
        label="Total fit"
    )

    ax0.plot(
        x,
        background,
        color="gray",
        linewidth=1.5,
        linestyle="--",
        label="Fitted continuum"
    )

    colors = {
        "Mn_Ka": "tab:blue",
        "Mn_Kb": "tab:blue",
        "Fe_Ka": "tab:brown",
        "Fe_Kb": "tab:brown"
    }

    for line, color in colors.items():

        peak = components[
            f"{line}_"
        ]

        ax0.plot(
            x,
            background + peak,
            color=color,
            linewidth=1.4,
            label=f"{line} + continuum"
        )

    ax0.set_ylabel(
        "Counts / channel"
    )

    ax0.set_title(
        "simultaneous EDS deconvolution"
    )

    ax0.legend(
        fontsize=8,
        ncol=2
    )

    residual = y - fit

    ax1.axhline(
        0,
        color="black",
        linewidth=0.8
    )

    ax1.step(
        x,
        residual,
        where="mid",
        color="black",
        linewidth=1
    )

    ax1.set_xlabel(
        "Energy (keV)"
    )

    ax1.set_ylabel(
        "Residual"
    )

    fig.tight_layout()

    return fig


# ============================================================
# Run the deconvolution
# ============================================================

result = fit_energy_eds(
    energy_keV=energy_keV,
    counts=counts,
    e_min=5.5,
    e_max=7.5,
    fwhm_eV=135.0,
    background="linear",
    allow_energy_shift=False,
)


# Extract peak areas
areas = fitted_peak_areas(
    result
)


# Print fit report and areas
print(
    result.fit_report()
)

display(
    areas
)


# Extract individual line areas
Mn_net = areas.loc[
    areas["line"].eq("Mn_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

Fe_net = areas.loc[
    areas["line"].eq("Fe_Ka"),
    "net_peak_area_count_keV"
].iloc[0]

print("Mn_net:", Mn_net)
print("Fe_net:", Fe_net)


# Obtain continuum-subtracted data
x_fit, y_fit, fitted_continuum, continuum_removed = (
    continuum_removed_spectrum(result)
)

print(
    "Continuum-subtracted spectrum generated:",
    continuum_removed.shape
)


# Plot fit (pop up gui)
fig = plot_energy_fit(
    result
)

plt.show()

[[Model]]
    ((((Model(linear, prefix='bg_') + Model(gaussian, prefix='Mn_Ka_')) + Model(gaussian, prefix='Mn_Kb_')) + Model(gaussian, prefix='Fe_Ka_')) + Model(gaussian, prefix='Fe_Kb_'))
[[Fit Statistics]]
    # fitting method   = least_squares
    # function evals   = 99
    # data points      = 200
    # variables        = 10
    chi-square         = 208.495105
    reduced chi-square = 1.09734266
    Akaike info crit   = 28.3196399
    Bayesian info crit = 61.3028136
    R-squared          = 0.97878481
##  Warning: uncertainties could not be estimated:
    Mn_Ka_sigma:      at boundary
    Mn_Kb_amplitude:  at boundary
    Mn_Kb_sigma:      at boundary
[[Variables]]
    bg_slope:        -267.164918 (init = -249.4975)
    bg_intercept:     2635.97799 (init = 2546.236)
    Mn_Ka_amplitude:  1.52682177 (init = 1)
    Mn_Ka_center:     5.893 (fixed)
    Mn_Ka_sigma:      0.10319260 (init = 0.05732922)
    Mn_Kb_amplitude:  3.0792e-11 (init = 20.06063)
    Mn_Kb_center:     6.491 (fixe

,line,net_peak_area_count_keV,net_peak_counts,area_standard_error,fitted_center_keV,sigma_eV,fwhm_eV
0,Mn_Ka,1.526822e+00,1.526822e+02,None,5.893,103.192601,243.000000
1,Mn_Kb,3.079213e-11,3.079213e-09,None,6.491,34.397534,81.000000
2,Fe_Ka,9.315737e+01,9.315737e+03,None,6.397,56.385249,132.777112
3,Fe_Kb,9.357242e+00,9.357242e+02,None,7.058,56.748136,133.631646


Mn_net: 1.526821773134234
Fe_net: 93.15737074392264
Continuum-subtracted spectrum generated: (200,)


In [44]:
# k-ratios corrected for the area fraction of the phase being quantified
std_df = pd.read_csv("15kV_Stds_Lib.csv")
std_row = std_df.iloc[0]

# Enter area fraction of the phase of interest (e.g., from the Otsu/multi-Otsu mask)
PHASE_AFRAC = 0.8301          # or: PHASE_AFRAC = C_C0_AFrac

if not np.isfinite(PHASE_AFRAC) or not (0 < PHASE_AFRAC <= 1):
    raise ValueError(f"Invalid phase area fraction ({PHASE_AFRAC}); must be in (0, 1].")

# Standard net counts (zero-based column positions in 15kV_Stds_Lib.csv)
std_net_counts = {
    "O":  pd.to_numeric(std_row.iloc[12], errors="raise"),
    "Na": pd.to_numeric(std_row.iloc[1],  errors="raise"),
    "Mg": pd.to_numeric(std_row.iloc[2],  errors="raise"),
    "Al": pd.to_numeric(std_row.iloc[3],  errors="raise"),
    "Si": pd.to_numeric(std_row.iloc[4],  errors="raise"),
    "Ca": pd.to_numeric(std_row.iloc[8],  errors="raise"),
    "Fe": pd.to_numeric(std_row.iloc[11], errors="raise")
}

unknown_net_counts = {
    "O":  O_net,
    "Na": Na_net,
    "Mg": Mg_net,
    "Al": Al_net,
    "Si": Si_net,
    "Ca": Ca_net,
    "Fe": Fe_net
}

# Every element must have both a standard and an unknown value
if set(std_net_counts) != set(unknown_net_counts):
    raise ValueError(
        f"Element mismatch: standards {sorted(std_net_counts)} vs "
        f"unknowns {sorted(unknown_net_counts)}"
    )

# Validate inputs
for element, i_std in std_net_counts.items():
    if not np.isfinite(i_std) or i_std <= 0:
        raise ValueError(f"{element}: invalid standard net count ({i_std}).")
    i_unknown = float(unknown_net_counts[element])
    if not np.isfinite(i_unknown) or i_unknown < 0:
        raise ValueError(f"{element}: invalid unknown net count ({i_unknown}).")

# k = I_unknown / (I_standard * area fraction)
kr_df = pd.DataFrame(
    {
        "element": list(std_net_counts.keys()),
        "k_ratio": [
            float(unknown_net_counts[el]) / (float(std_net_counts[el]) * PHASE_AFRAC)
            for el in std_net_counts
        ],
    }
)

print(f"Measured k-ratios (phase area fraction = {PHASE_AFRAC:.4f})")
display(kr_df.style.format({"k_ratio": "{:.6f}"}))

# Export for NeXL
input_csv = Path("input_kratios.csv").resolve()
kr_df[["element", "k_ratio"]].to_csv(input_csv, index=False, float_format="%.6f")
print(f"Saved: {input_csv}")

Measured k-ratios (phase area fraction = 0.8301)


,element,k_ratio
0,O,1.441733
1,Na,1.328848
2,Mg,0.007260
3,Al,1.382310
4,Si,0.851795
5,Ca,0.018533
6,Fe,0.006394


Saved: /Users/thomaslam/Downloads/input_kratios.csv


In [45]:
from pathlib import Path

workdir = Path("/Users/thomaslam/Documents/NeXL_XPP")
workdir.mkdir(exist_ok=True)

input_csv = workdir / "input_kratios.csv"
kr_df.to_csv(input_csv, index=False)

print(f"Saved NeXL input: {input_csv}")
print(kr_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

Saved NeXL input: /Users/thomaslam/Documents/NeXL_XPP/input_kratios.csv
element  k_ratio
      O 1.441733
     Na 1.328848
     Mg 0.007260
     Al 1.382310
     Si 0.851795
     Ca 0.018533
     Fe 0.006394


In [46]:
from pathlib import Path
import subprocess
import textwrap
import pandas as pd

# ================================================================
# 1. Python-controlled inputs
# ================================================================
BEAM_ENERGY_KEV = 15.0
TAKE_OFF_ANGLE_DEG = 35.0

input_csv = Path(input_csv).resolve()
standards_csv = Path("/Users/thomaslam/Documents/NeXL_XPP/Standards_df.csv")

project_dir = Path("/Users/thomaslam/Documents/NeXL_XPP")
project_dir.mkdir(parents=True, exist_ok=True)

results_csv = project_dir / "xpp_results.csv"
julia_script = project_dir / "run_xpp_from_python.jl"

# ================================================================
# 2. Validate Python inputs and write k-ratio CSV
# ================================================================
if not input_csv.is_file():
    raise FileNotFoundError(f"Python cannot find:\n{input_csv}")

if not standards_csv.is_file():
    raise FileNotFoundError(f"Python cannot find:\n{standards_csv}")

kr_df = kr_df.copy()
kr_df.columns = kr_df.columns.astype(str).str.strip()

required_columns = {"element", "k_ratio"}
if not required_columns.issubset(set(kr_df.columns)):
    raise ValueError(
        f"kr_df must contain {required_columns}; found {list(kr_df.columns)}"
    )

kr_df["element"] = kr_df["element"].astype(str).str.strip()
kr_df["k_ratio"] = pd.to_numeric(kr_df["k_ratio"], errors="raise")

if kr_df["element"].duplicated().any():
    duplicates = kr_df.loc[
        kr_df["element"].duplicated(keep=False), "element"
    ].tolist()
    raise ValueError(
        "Each element must appear only once in kr_df. "
        f"Duplicate entries: {duplicates}"
    )

if (kr_df["k_ratio"] < 0).any():
    raise ValueError("All k_ratio values must be non-negative.")

print("Python input k-ratios:")
display(kr_df[["element", "k_ratio"]])

kr_df[["element", "k_ratio"]].to_csv(input_csv, index=False)

# ================================================================
# 3. Julia backend written by Python
# ================================================================
julia_code = r'''
using CSV
using DataFrames
using NeXLMatrixCorrection

input_kratios_csv = ARGS[1]
standards_csv = ARGS[2]
results_csv = ARGS[3]
beam_energy_kev = parse(Float64, ARGS[4])
take_off_angle_deg = parse(Float64, ARGS[5])

kr_df = CSV.read(input_kratios_csv, DataFrame)
standards_df = CSV.read(standards_csv, DataFrame)

required_columns = ["element", "k_ratio"]
found_columns = String.(names(kr_df))
missing_columns = setdiff(required_columns, found_columns)

isempty(missing_columns) || error(
    "input_kratios.csv missing columns: $(missing_columns). " *
    "Found: $(found_columns)"
)

println(
    "Quantification conditions from Python: E0 = ", beam_energy_kev,
    " keV; TOA = ", take_off_angle_deg, " degrees"
)

unknown_properties = Dict(
    :BeamEnergy => beam_energy_kev * 1.0e3,
    :TakeOffAngle => deg2rad(take_off_angle_deg)
)

standard_properties = Dict(
    :BeamEnergy => beam_energy_kev * 1.0e3,
    :TakeOffAngle => deg2rad(take_off_angle_deg)
)

const ATOMIC_WEIGHT = Dict(
    "O"  => 15.999,
    "Na" => 22.990,
    "Mg" => 24.305,
    "Al" => 26.982,
    "Si" => 28.086,
    "K"  => 39.098,
    "Ca" => 40.078,
    "Ti" => 47.867,
    "Mn" => 54.938,
    "Fe" => 55.845
)

const OXIDE_DEFINITIONS = Dict(
    :SiO2  => ("Si", 1, 2),
    :TiO2  => ("Ti", 1, 2),
    :Al2O3 => ("Al", 2, 3),
    :FeO   => ("Fe", 1, 1),
    :Fe2O3 => ("Fe", 2, 3),
    :MnO   => ("Mn", 1, 1),
    :MgO   => ("Mg", 1, 1),
    :CaO   => ("Ca", 1, 1),
    :Na2O  => ("Na", 2, 1),
    :K2O   => ("K", 2, 1)
)

function elemental_fractions_from_oxides(row)
    elemental_wtpercent = Dict{String, Float64}()

    for (oxide, (element, n_element, n_oxygen)) in OXIDE_DEFINITIONS
        oxide in propertynames(row) || continue

        wtpercent = row[oxide]

        if ismissing(wtpercent) || Float64(wtpercent) <= 0.0
            continue
        end

        wtpercent = Float64(wtpercent)

        molecular_weight =
            n_element * ATOMIC_WEIGHT[element] +
            n_oxygen * ATOMIC_WEIGHT["O"]

        elemental_wtpercent[element] =
            get(elemental_wtpercent, element, 0.0) +
            wtpercent * n_element * ATOMIC_WEIGHT[element] / molecular_weight

        elemental_wtpercent["O"] =
            get(elemental_wtpercent, "O", 0.0) +
            wtpercent * n_oxygen * ATOMIC_WEIGHT["O"] / molecular_weight
    end

    total = sum(values(elemental_wtpercent))

    total > 0.0 || error(
        "No positive oxide wt% values found for standard: " *
        String(row.standard_id)
    )

    return Dict(
        element => wtpercent / total
        for (element, wtpercent) in elemental_wtpercent
    )
end

function get_standard_fractions(standard_id)
    hits = filter(
        row -> strip(String(row.standard_id)) == standard_id,
        standards_df
    )

    nrow(hits) == 1 || error(
        "Expected exactly one standard row for \"" * standard_id *
        "\"; found $(nrow(hits))."
    )

    return elemental_fractions_from_oxides(hits[1, :])
end

function make_material(standard_id)
    fractions = get_standard_fractions(standard_id)

    composition = join(
        [
            string(value, "*", element)
            for (element, value) in sort(collect(fractions); by=first)
        ],
        "+"
    )

    return parse(Material, composition; name=standard_id)
end

standards = Dict(
    "O"  => make_material("orthoclase_spi_mci"),
    "Na" => make_material("albite_spi_mci"),
    "Mg" => make_material("MgO_spi_mci"),
    "Al" => make_material("orthoclase_spi_mci"),
    "Si" => make_material("orthoclase_spi_mci"),
    "K" => make_material("orthoclase_spi_mci"),
    "Ca" => make_material("wollastonite_spi_mci"),
    "Mn" => make_material("rhodonite_spi_mci"),
    "Fe" => make_material("hematite_spi_mci")
)

lines = Dict(
    "O"  => n"O K-L3",
    "Na" => n"Na K-L3",
    "Mg" => n"Mg K-L3",
    "Al" => n"Al K-L3",
    "Si" => n"Si K-L3",
    "K" => n"K K-L3",
    "Ca" => n"Ca K-L3",
    "Mn" => n"Mn K-L3",
    "Fe" => n"Fe K-L3"
)

krs = KRatio[]

for row in eachrow(kr_df)
    element = strip(String(row.element))

    haskey(standards, element) || error(
        "No standard mapping for element: \"$element\""
    )

    k = Float64(row.k_ratio)
    k >= 0.0 || error("Negative k-ratio for $element")

    # Uses a 1% relative k-ratio uncertainty, with a small positive floor.
    sigma_k = max(0.01 * max(k, 1.0e-6), 1.0e-6)

    push!(
        krs,
        KRatio(
            [lines[element]],
            unknown_properties,
            standard_properties,
            standards[element],
            uv(k, sigma_k)
        )
    )
end

println("K-ratios received from Python:")
foreach(println, krs)

# NeXL quantification has completed successfully in this environment.
result = quantify(label("python_xpp"), krs)

# ----------------------------------------------------------------
# Export directly from IterationResult; do NOT call asa(DataFrame,...)
# because the installed NeXL DataFrames extension references _ZAF,
# which is undefined in that package version.
# ----------------------------------------------------------------

# `result.comp` is the final quantified Material. Build the CSV in
# input-element order, so Python receives a predictable tidy table.
result_elements = String[]
mass_fractions = Float64[]

for row in eachrow(kr_df)
    element_name = strip(String(row.element))

    # `parse(Element, ...)` creates the element key used by Material.
    element = parse(Element, element_name)

    # The Material is indexed by Element and returns the elemental
    # mass fraction. This bypasses NeXL's broken DataFrames extension.
    mass_fraction = Float64(result.comp[element])

    push!(result_elements, element_name)
    push!(mass_fractions, mass_fraction)
end

out_df = DataFrame(
    label = fill(string(result.label), length(result_elements)),
    converged = fill(Bool(result.converged), length(result_elements)),
    iterations = fill(Int(result.iterations), length(result_elements)),
    beam_energy_kev = fill(beam_energy_kev, length(result_elements)),
    take_off_angle_deg = fill(take_off_angle_deg, length(result_elements)),
    element = result_elements,
    mass_fraction = mass_fractions,
    wt_percent = 100.0 .* mass_fractions
)

CSV.write(results_csv, out_df)

println("Final quantification result: ", result)
println("Wrote results to: ", results_csv)
'''

julia_script.write_text(
    textwrap.dedent(julia_code),
    encoding="utf-8"
)

# ================================================================
# 4. Python launches Julia and reads results back
# ================================================================
run = subprocess.run(
    [
        "julia",
        str(julia_script),
        str(input_csv),
        str(standards_csv),
        str(results_csv),
        str(BEAM_ENERGY_KEV),
        str(TAKE_OFF_ANGLE_DEG),
    ],
    text=True,
    capture_output=True,
)

print("----- Julia stdout -----")
print(run.stdout)

if run.returncode != 0:
    print("----- Julia stderr -----")
    print(run.stderr)
    raise RuntimeError("Julia/NeXL quantification did not complete.")

if not results_csv.is_file():
    raise FileNotFoundError(
        f"Julia did not create the results output:\n{results_csv}"
    )

# ----- Load the Julia/NeXL results CSV back into Python -----
xpp_results = pd.read_csv(results_csv)

print("----- Results returned to Python -----")
display(xpp_results)

print("\n----- Quantified elemental total -----")
print(f"{xpp_results['mass_fraction'].sum():.6f}")

Python input k-ratios:


,element,k_ratio
0,O,1.441733
1,Na,1.328848
2,Mg,0.007260
3,Al,1.382310
4,Si,0.851795
5,Ca,0.018533
6,Fe,0.006394


----- Julia stdout -----
Quantification conditions from Python: E0 = 15.0 keV; TOA = 35.0 degrees
K-ratios received from Python:
k[O K-L3, orthoclase_spi_mci] = 1.442 ± 0.014
k[Na K-L3, albite_spi_mci] = 1.329 ± 0.013
k[Mg K-L3, MgO_spi_mci] = 7.26e-03 ± 7.3e-05
k[Al K-L3, orthoclase_spi_mci] = 1.382 ± 0.014
k[Si K-L3, orthoclase_spi_mci] = 0.8518 ± 0.0085
k[Ca K-L3, wollastonite_spi_mci] = 0.0185 ± 0.0002
k[Fe K-L3, hematite_spi_mci] = 6.39e-03 ± 6.4e-05
Final quantification result: Converged to python_xpp[O=0.4699,Na=0.1128,Mg=0.0055,Al=0.1310,Si=0.2807,Ca=0.0067,Fe=0.0051] in 11 steps.
Wrote results to: /Users/thomaslam/Documents/NeXL_XPP/xpp_results.csv

----- Results returned to Python -----


,label,converged,iterations,beam_energy_kev,take_off_angle_deg,element,mass_fraction,wt_percent
0,python_xpp,True,11,15.0,35.0,O,0.469859,46.985878
1,python_xpp,True,11,15.0,35.0,Na,0.112850,11.284987
2,python_xpp,True,11,15.0,35.0,Mg,0.005462,0.546154
3,python_xpp,True,11,15.0,35.0,Al,0.130973,13.097334
4,python_xpp,True,11,15.0,35.0,Si,0.280650,28.065023
5,python_xpp,True,11,15.0,35.0,Ca,0.006671,0.667135
6,python_xpp,True,11,15.0,35.0,Fe,0.005094,0.509409



----- Quantified elemental total -----
1.011559


In [47]:
# ================================================================
# 5. Convert quantified elemental mass fractions to oxide equivalents
# ================================================================

# Select the reporting convention for iron:
# "FeO"   = report all quantified Fe as ferrous iron
# "Fe2O3" = report all quantified Fe as ferric iron
IRON_OXIDE = "FeO"

# Atomic weights in g/mol
ATOMIC_WEIGHT = {
    "O": 15.999,
    "Na": 22.990,
    "Mg": 24.305,
    "Al": 26.982,
    "Si": 28.086,
    "Ca": 40.078,
    "Fe": 55.845,
}

# Oxide stoichiometry:
# oxide_name: (cation_symbol, number_of_cations, number_of_oxygens)
OXIDE_DEFINITIONS = {
    "Na2O": ("Na", 2, 1),
    "Al2O3": ("Al", 2, 3),
    "MgO": ("Mg", 1, 1),
    "SiO2": ("Si", 1, 2),
    "CaO": ("Ca", 1, 1),
    IRON_OXIDE: ("Fe", 1, 1) if IRON_OXIDE == "FeO" else ("Fe", 2, 3),
}

# Element wt% -> oxide wt% factor:
# factor = molar_mass(oxide) / mass_of_cations_in_one_formula_unit
oxide_rows = []

for oxide, (element, n_cation, n_oxygen) in OXIDE_DEFINITIONS.items():
    match = xpp_results.loc[
        xpp_results["element"].astype(str).str.strip() == element
    ]

    if match.empty:
        print(f"Warning: no quantified {element} row; skipping {oxide}.")
        continue

    elemental_mass_fraction = float(match["mass_fraction"].iloc[0])
    elemental_wt_percent = 100.0 * elemental_mass_fraction

    oxide_molar_mass = (
        n_cation * ATOMIC_WEIGHT[element]
        + n_oxygen * ATOMIC_WEIGHT["O"]
    )

    cation_mass_per_formula_unit = n_cation * ATOMIC_WEIGHT[element]

    element_to_oxide_factor = (
        oxide_molar_mass / cation_mass_per_formula_unit
    )

    oxide_wt_percent = elemental_wt_percent * element_to_oxide_factor

    oxide_rows.append(
        {
            "oxide": oxide,
            "source_element": element,
            "element_mass_fraction": elemental_mass_fraction,
            "element_wt_percent": elemental_wt_percent,
            "element_to_oxide_factor": element_to_oxide_factor,
            "oxide_wt_percent": oxide_wt_percent,
            "oxide_mass_fraction": oxide_wt_percent / 100.0,
        }
    )

oxide_results = pd.DataFrame(oxide_rows)

print("----- Oxide-equivalent results (unnormalized) -----")
display(oxide_results)

oxide_total_wt_percent = oxide_results["oxide_wt_percent"].sum()

print("\n----- Oxide total -----")
print(f"{oxide_total_wt_percent:.2f} wt%")

# save the oxide table beside the elemental XPP results.
oxide_results_csv = project_dir / "C_C0_xpp_oxide_results.csv"
oxide_results.to_csv(oxide_results_csv, index=False)

print(f"\nWrote oxide results to: {oxide_results_csv}")


----- Oxide-equivalent results (unnormalized) -----


,oxide,source_element,element_mass_fraction,element_wt_percent,element_to_oxide_factor,oxide_wt_percent,oxide_mass_fraction
0,Na2O,Na,0.112850,11.284987,1.347956,15.211662,0.152117
1,Al2O3,Al,0.130973,13.097334,1.889426,24.746447,0.247464
2,MgO,Mg,0.005462,0.546154,1.658260,0.905665,0.009057
3,SiO2,Si,0.280650,28.065023,2.139286,60.039125,0.600391
4,CaO,Ca,0.006671,0.667135,1.399197,0.933454,0.009335
5,FeO,Fe,0.005094,0.509409,1.286489,0.655350,0.006553



----- Oxide total -----
102.49 wt%

Wrote oxide results to: /Users/thomaslam/Documents/NeXL_XPP/C_C0_xpp_oxide_results.csv
